# Frozen confirmation and final fitting v005

This notebook fits no A candidates. It freezes a complete, predeclared A snapshot,
then refits only the incumbent, selected individual and bounded pair on grouped C.
Original A and C are excluded from every C fit and learned transformation. C is a
resampling/filtering stage and never controls stopping, schedules or pair weights.

Default mode reviews a valid saved round. Set `PA3_RUN_CONFIRM_V005=1` to execute,
`PA3_RESUME_CONFIRM_V005=1` to reuse compatible states, and
`PA3_V005_CONFIRM_LIMIT=lock`, `C`, or `final`. Choose a completed source phase with
`PA3_V005_CONFIRM_A_PHASE`; each new A lock uses a distinct round name.

Final fits use all 67,905 approved labeled breaths with frozen A budgets. Earlier
A/C scores describe their validation models; final models have no independent
holdout. CSV upload is a separate action. Original source and archives are retained.


## Configuration and review-safe helpers

In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import gc, hashlib, json, time, copy, uuid, shutil, re, fcntl, textwrap
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from IPython.display import display
RUN = os.environ.get('PA3_RUN_CONFIRM_V005', '0') == '1'
RESUME = os.environ.get('PA3_RESUME_CONFIRM_V005', '0') == '1'
LIMIT = os.environ.get('PA3_V005_CONFIRM_LIMIT', 'lock')
ROUND = os.environ.get('PA3_V005_CONFIRM_ROUND', 'round1')
A_PHASE = os.environ.get('PA3_V005_CONFIRM_A_PHASE', 'features')
PHASES = ['schedule', 'features', 'capacity', 'refinement', 'seeds']
assert LIMIT in ['lock', 'C', 'final'] and A_PHASE in PHASES
assert re.fullmatch(r'[A-Za-z0-9_-]+', ROUND)
PROJECT = Path(os.environ.get('PA3_PROJECT', Path.cwd())).resolve()
if not (PROJECT/'data').is_dir() and (PROJECT/'ventilator-pressure-prediction/data').is_dir():
    PROJECT = PROJECT/'ventilator-pressure-prediction'
DATA = PROJECT/'data'
SEQ = PROJECT/'outputs/improvements_v005/sequence'
TREE = PROJECT/'outputs/improvements_v005/tree_context'
OLD = PROJECT/'outputs/sequence_v004b'
V3 = PROJECT/'outputs/controlled_v003'
OUT = PROJECT/'outputs/improvements_v005'/('confirmation_'+ROUND)
FINAL = OUT/'final_all'
THREADS = 4
MIN_GAIN = 1e-6
DTYPES = dict(id='int64', breath_id='int32', R='int8', C='int8',
    time_step='float32', u_in='float32', u_out='int8', pressure='float32')
INPUT_COLUMNS = ['id', 'breath_id', 'R', 'C', 'time_step', 'u_in', 'u_out']
PLAN = dict(version='confirmation_final_v005-r1', round=ROUND, A_snapshot_phase=A_PHASE,
    weights=[.25,.5,.75], pair_search='individual models only; no nested blends',
    A_confirmation_approaches=['fixed incumbent blend','locked A-best individual','locked A-best pair'],
    C_fit_breaths=54323, C_validation_breaths=6791, A_fit_breaths=61114,
    A_validation_breaths=6791, final_breaths=67905, threads=THREADS,
    rule='Reference always eligible. Locked individual must beat reference on A and C. '
         'Locked pair must beat the locked A-best individual on A and C and reference on C. '
         'Choose lowest-A-MAE eligible approach. No runner-up/weight search after C.',
    C_is_resampling_confirmation=True, C_stopping_or_scheduler_feedback=False,
    final_has_independent_holdout=False, automatic_submission=False,
    legacy_C='fresh fused-v004 GRU base99 A-rates then29 constant1e-5 freshAdam; fresh2500-tree511')

def canonical_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()

def digest(path):
    result = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''):
            result.update(block)
    return result.hexdigest()

def array_hash(value):
    return hashlib.sha256(np.ascontiguousarray(value).tobytes()).hexdigest()

def code_hash(path):
    nb = json.loads(Path(path).read_text())
    return hashlib.sha256('\n'.join(''.join(c['source']) for c in nb['cells']
        if c['cell_type']=='code').encode()).hexdigest()

def write_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name+'.tmp-'+uuid.uuid4().hex)
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False)+'\n')
    os.replace(temporary, path)

def save_array(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name+'.tmp-'+uuid.uuid4().hex)
    with temporary.open('wb') as stream:
        np.save(stream, value, allow_pickle=False)
    os.replace(temporary, path)

def resolve_artifact(value):
    value = str(value)
    marker = '/ventilator-pressure-prediction/'
    if marker in value:
        result = PROJECT/value.split(marker, 1)[1]
    else:
        result = Path(value) if Path(value).is_absolute() else PROJECT/value
    result = result.resolve()
    if not result.is_relative_to(PROJECT):
        raise ValueError('Artifact is outside the approved project: '+value)
    return result

def event(kind, **values):
    with (OUT/'events.jsonl').open('a') as stream:
        stream.write(json.dumps(dict(timestamp_utc=datetime.now(timezone.utc).isoformat(),
            event=kind, **values), allow_nan=False)+'\n')

def preserve_partial(paths, label):
    present = [Path(p) for p in paths if Path(p).exists()]
    if not present:
        return
    destination = OUT/'archive'/(label+'_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
    destination.mkdir(parents=True, exist_ok=False)
    for path in present:
        shutil.move(str(path), str(destination/path.name))
    event('interrupted_artifacts_preserved', label=label, directory=str(destination))

class OneModeFeatureCache:
    def __init__(self, maker=None):
        self.maker = maker
        self.key = self.value = None
    def clear(self):
        self.key = self.value = None
        gc.collect()
    def get(self, frame, mode, generation):
        key = (generation, mode)
        if key != self.key:
            self.clear()
            maker = make_sequence_features if self.maker is None else self.maker
            self.value = maker(frame, mode)
            self.key = key
        return self.value

def aligned_mae(prediction, target, row_ids, expected_ids):
    prediction, target = np.asarray(prediction,dtype=np.float64), np.asarray(target,dtype=np.float64)
    if not np.array_equal(row_ids, expected_ids):
        raise ValueError('Scored row ID order differs')
    if prediction.ndim != 1 or prediction.shape != target.shape or not np.isfinite(prediction).all() or not np.isfinite(target).all():
        raise ValueError('Invalid aligned prediction shape/values')
    return float(np.abs(prediction-target).mean())

def choose_A(candidates, predictions, target, reference_members, reference_weights):
    unique, seen = [], set()
    for record in sorted(candidates,key=lambda r:r['name']):
        key = (record['checkpoint_sha256'], array_hash(predictions[record['name']]))
        if key not in seen:
            unique.append(record); seen.add(key)
    if not unique:
        raise ValueError('No complete A individuals')
    best = min(unique,key=lambda r:(r['A_mae'],r['name']))
    pairs = []
    for index, first in enumerate(unique):
        for second in unique[index+1:]:
            for weight in PLAN['weights']:
                p = weight*predictions[first['name']] + (1-weight)*predictions[second['name']]
                pairs.append(dict(members=[first['name'],second['name']],weights=[weight,1-weight],
                    A_mae=float(np.abs(p-target).mean())))
    if not pairs:
        raise ValueError('At least two distinct A model individuals are required')
    pair = min(pairs,key=lambda r:(r['A_mae'],r['members'],r['weights']))
    reference = sum(w*predictions[n] for n,w in zip(reference_members,reference_weights))
    return dict(best_individual=dict(members=[best['name']],weights=[1.],A_mae=best['A_mae']),
        best_pair=pair, reference=dict(members=reference_members,weights=reference_weights,
            A_mae=float(np.abs(reference-target).mean()))), pairs

def choose_C(frozen_approaches, c_predictions, c_target):
    table = []
    for name in ['reference','best_individual','best_pair']:
        a = frozen_approaches[name]
        p = sum(w*c_predictions[n] for n,w in zip(a['members'],a['weights']))
        table.append(dict(approach=name,**a,C_mae=float(np.abs(p-c_target).mean())))
    reference, individual, pair = table
    reference['eligible'] = True
    individual['eligible'] = bool(individual['A_mae'] < reference['A_mae']-MIN_GAIN
        and individual['C_mae'] < reference['C_mae']-MIN_GAIN)
    pair['eligible'] = bool(pair['A_mae'] < individual['A_mae']-MIN_GAIN
        and pair['C_mae'] < individual['C_mae']-MIN_GAIN
        and pair['C_mae'] < reference['C_mae']-MIN_GAIN)
    selected = min([r for r in table if r['eligible']],key=lambda r:(r['A_mae'],r['approach']))
    return table, selected

print('Mode:', 'execute confirmation' if RUN else 'review saved confirmation', '| round:', ROUND,
      '| phase limit:', LIMIT, '| A snapshot:', A_PHASE)
print('Artifacts:', OUT)

## Embedded approved-input and sequence definitions

In [ ]:
if RUN:
    import hashlib
    import json
    from pathlib import Path

    import numpy as np
    import pandas as pd

    STEPS = 80
    V005_FEATURE_MODES = ('causal20', 'offline', 'dynamics', 'combined')
    V005_BASE_FEATURE_NAMES = (
        'R', 'C', 'time_step', 'u_in', 'u_out', 'step', 'dt',
        'control_sum', 'control_mean', 'control_integral',
        'u_in_lag_1', 'u_in_change_1', 'u_in_lag_2', 'u_in_change_2',
        'u_in_lag_4', 'u_in_change_4', 'u_in_lag_8', 'u_in_change_8',
        'u_in_mean_4', 'u_in_mean_8',
    )
    V005_OFFLINE_FEATURE_NAMES = (
        'u_in_lead_1', 'u_in_lead_2', 'u_in_lead_4', 'u_in_lead_8',
        'u_in_centered_mean_4', 'u_in_centered_mean_8',
        'inspiration_duration', 'inspiration_time_fraction',
        'time_to_u_out_switch', 'inspiration_u_in_peak',
        'inspiration_u_in_peak_time', 'inspiration_u_in_mean',
        'inspiration_u_in_std', 'inspiration_u_in_integral',
    )
    V005_EWM_TIMESCALES_SECONDS = (.05, .1, .2, .5, 1., 2.)
    V005_DYNAMICS_FEATURE_NAMES = (
        'rc_tau_seconds', 'time_over_rc_tau', 'u_in_times_R', 'u_in_over_R',
        'du_in_dt', 'control_trapezoid_integral',
        'control_trapezoid_over_compliance_L',
        'control_ewm_tau_0_05', 'control_ewm_tau_0_1',
        'control_ewm_tau_0_2', 'control_ewm_tau_0_5',
        'control_ewm_tau_1', 'control_ewm_tau_2',
    )

In [ ]:
if RUN:
    def features(frame: pd.DataFrame) -> tuple[np.ndarray, list[str], np.ndarray]:
        """Return features in original row order, after verifying 80-row blocks."""
        if len(frame) % STEPS:
            raise ValueError("Unexpected breath length; do not reshape this file")
        blocks = frame["breath_id"].to_numpy().reshape(-1, STEPS)
        breath_ids = blocks[:, 0]
        if not np.all(blocks == breath_ids[:, None]):
            raise ValueError("Breaths are not contiguous 80-row blocks")
        if len(np.unique(breath_ids)) != len(breath_ids):
            raise ValueError("Repeated breath blocks")
        if not frame["id"].is_unique:
            raise ValueError("Duplicate row IDs")
        u = frame["u_in"].to_numpy().reshape(-1, STEPS)
        t = frame["time_step"].to_numpy().reshape(-1, STEPS)
        if not np.all(np.diff(t, axis=1) > 0):
            raise ValueError("Timestamps must increase inside each breath")
        for column in ("R", "C"):
            values = frame[column].to_numpy().reshape(-1, STEPS)
            if not np.all(values == values[:, :1]):
                raise ValueError(f"{column} changes within a breath")

        columns = {}
        for column in ("R", "C", "time_step", "u_in", "u_out"):
            columns[column] = frame[column].to_numpy(dtype=np.float32)
        step = np.arange(STEPS)
        dt = np.diff(t, axis=1, prepend=t[:, :1])
        cumulative = np.cumsum(u, axis=1, dtype=np.float32)
        columns["step"] = np.broadcast_to(step, u.shape).ravel()
        columns["dt"] = dt.ravel()
        columns["control_sum"] = cumulative.ravel()
        columns["control_mean"] = (cumulative / (step + 1)).ravel()
        # Valve opening is not measured flow: this is a history proxy, not volume.
        columns["control_integral"] = np.cumsum(u * dt, axis=1).ravel()
        for lag in (1, 2, 4, 8):
            shifted = np.empty_like(u)
            shifted[:, :lag] = u[:, :1]
            shifted[:, lag:] = u[:, :-lag]
            columns[f"u_in_lag_{lag}"] = shifted.ravel()
            columns[f"u_in_change_{lag}"] = (u - shifted).ravel()
        padded_sum = np.concatenate([np.zeros((len(u), 1), dtype=np.float32), cumulative], axis=1)
        for window in (4, 8):
            begin = np.maximum(0, step + 1 - window)
            columns[f"u_in_mean_{window}"] = (
                (padded_sum[:, step + 1] - padded_sum[:, begin]) / (step + 1 - begin)
            ).ravel()
        names = list(columns)
        matrix = np.column_stack(list(columns.values())).astype(np.float32, copy=False)
        if not np.isfinite(matrix).all() or "pressure" in names:
            raise ValueError("Invalid features")
        return matrix, names, breath_ids

In [ ]:
if RUN:
    # Keep the legacy flat API available when importing these cells into a notebook.
    features_causal20 = features


    def _validate_feature_frame_v005(frame):
        """Validate known inputs only. Pressure and unrelated columns are never read."""
        if not isinstance(frame, pd.DataFrame):
            raise TypeError('Expected a pandas DataFrame')
        required = ('id', 'breath_id', 'R', 'C', 'time_step', 'u_in', 'u_out')
        if not frame.columns.is_unique:
            raise ValueError('Duplicate column names')
        missing = set(required) - set(frame.columns)
        if missing:
            raise ValueError(f'Missing required inputs: {sorted(missing)}')
        if not len(frame) or len(frame) % STEPS:
            raise ValueError('Expected nonempty contiguous 80-row breaths')
        for column in required:
            if not pd.api.types.is_numeric_dtype(frame[column]):
                raise ValueError(f'{column} must be numeric')
            values = frame[column].to_numpy()
            if not np.isfinite(values).all():
                raise ValueError(f'{column} must be finite')
        for column in ('id', 'breath_id'):
            values = frame[column].to_numpy()
            if not np.all(values == np.floor(values)):
                raise ValueError(f'{column} must contain integer IDs')
        if not frame['id'].is_unique:
            raise ValueError('Duplicate row IDs')
        blocks = frame['breath_id'].to_numpy().reshape(-1, STEPS)
        breath_ids = blocks[:, 0]
        if not np.all(blocks == breath_ids[:, None]):
            raise ValueError('Breaths are not contiguous 80-row blocks')
        if len(np.unique(breath_ids)) != len(breath_ids):
            raise ValueError('Repeated breath blocks')
        t = frame['time_step'].to_numpy(dtype=np.float64).reshape(-1, STEPS)
        if not np.all(np.diff(t, axis=1) > 0):
            raise ValueError('Timestamps must increase inside each breath')
        for column, levels in (('R', (5, 20, 50)), ('C', (10, 20, 50))):
            values = frame[column].to_numpy().reshape(-1, STEPS)
            if not np.all(values == values[:, :1]):
                raise ValueError(f'{column} changes within a breath')
            if not np.isin(values, levels).all():
                raise ValueError(f'Unexpected {column} level')
        out = frame['u_out'].to_numpy().reshape(-1, STEPS)
        if not np.isin(out, (0, 1)).all():
            raise ValueError('u_out must be binary')
        # The approved data has a single inspiration followed by expiration. A fully
        # observed inspiration without an expiration sample is supported as well.
        if np.any(out[:, 0] != 0) or np.any(np.diff(out.astype(np.int8), axis=1) < 0):
            raise ValueError('Expected initial inspiration followed by at most one valve opening')
        return breath_ids


    def feature_names_v005(feature_mode='causal20'):
        if feature_mode not in V005_FEATURE_MODES:
            raise ValueError(f'Unknown feature_mode: {feature_mode!r}')
        names = list(V005_BASE_FEATURE_NAMES)
        if feature_mode in ('offline', 'combined'):
            names.extend(V005_OFFLINE_FEATURE_NAMES)
        if feature_mode in ('dynamics', 'combined'):
            names.extend(V005_DYNAMICS_FEATURE_NAMES)
        return names


    def _offline_feature_block_v005(frame):
        """Known whole-breath input context; no target values or statistics."""
        u = frame['u_in'].to_numpy(dtype=np.float64).reshape(-1, STEPS)
        t = frame['time_step'].to_numpy(dtype=np.float64).reshape(-1, STEPS)
        out = frame['u_out'].to_numpy().reshape(-1, STEPS)
        inspiration = out == 0
        n_breaths = len(u)
        columns = []
        for lead in (1, 2, 4, 8):
            shifted = np.empty_like(u)
            shifted[:, :-lead] = u[:, lead:]
            shifted[:, -lead:] = u[:, -1:]
            columns.append(shifted)
        prefix = np.concatenate((np.zeros((n_breaths, 1)), np.cumsum(u, axis=1)), axis=1)
        steps = np.arange(STEPS)
        for window in (4, 8):
            # Even centered windows have window//2 prior and window//2-1 future
            # neighbours, matching pandas rolling(center=True, min_periods=1).
            begin = np.maximum(0, steps - window//2)
            end = np.minimum(STEPS, steps + (window+1)//2)
            columns.append((prefix[:, end] - prefix[:, begin]) / (end-begin))
        has_expiration = np.any(out == 1, axis=1)
        first_expiration = np.argmax(out == 1, axis=1)
        switch_index = np.where(has_expiration, first_expiration, STEPS-1)
        switch_time = t[np.arange(n_breaths), switch_index]
        duration = switch_time - t[:, 0]
        # With strictly increasing 80-step timestamps and initial inspiration,
        # duration is positive, including the no-expiration boundary case.
        fraction = np.clip((t-t[:, :1])/duration[:, None], 0., 1.)
        until_switch = np.maximum(switch_time[:, None]-t, 0.)
        masked = np.where(inspiration, u, -np.inf)
        peak_index = np.argmax(masked, axis=1)  # first peak resolves a known-input tie
        peak = u[np.arange(n_breaths), peak_index]
        peak_time = t[np.arange(n_breaths), peak_index] - t[:, 0]
        count = inspiration.sum(axis=1)
        mean = np.sum(np.where(inspiration, u, 0.), axis=1)/count
        variance = np.sum(np.where(inspiration, (u-mean[:, None])**2, 0.), axis=1)/count
        std = np.sqrt(np.maximum(variance, 0.))
        # Integrate between inspiratory observations only; do not infer unobserved
        # flow or include the interval crossing into expiration.
        both_inspiration = inspiration[:, 1:] & inspiration[:, :-1]
        integral = np.sum(np.where(both_inspiration,
            .5*(u[:, 1:]+u[:, :-1])*np.diff(t, axis=1), 0.), axis=1)
        def broadcast(values):
            return np.broadcast_to(values[:, None], u.shape)
        columns.extend((broadcast(duration), fraction, until_switch, broadcast(peak),
            broadcast(peak_time), broadcast(mean), broadcast(std), broadcast(integral)))
        return np.stack(columns, axis=-1).astype(np.float32)


    def _dynamics_feature_block_v005(frame):
        """Causal timing and control proxies; valve opening is not measured flow."""
        u = frame['u_in'].to_numpy(dtype=np.float64).reshape(-1, STEPS)
        t = frame['time_step'].to_numpy(dtype=np.float64).reshape(-1, STEPS)
        r = frame['R'].to_numpy(dtype=np.float64).reshape(-1, STEPS)
        c_litres = frame['C'].to_numpy(dtype=np.float64).reshape(-1, STEPS)/1000.
        dt = np.diff(t, axis=1, prepend=t[:, :1])
        tau = r*c_litres  # R in cmH2O/(L/s), C converted from mL/cmH2O
        derivative = np.zeros_like(u)
        derivative[:, 1:] = np.diff(u, axis=1)/dt[:, 1:]
        trapezoid = np.zeros_like(u)
        trapezoid[:, 1:] = np.cumsum(.5*(u[:, 1:]+u[:, :-1])*dt[:, 1:], axis=1)
        # These interactions can help the learner represent R/C sensitivity. They
        # are not asserted to have physical pressure/flow/volume units because u_in
        # is a valve-opening percentage, not observed flow or driving pressure.
        columns = [tau, (t-t[:, :1])/tau, u*r, u/r, derivative,
                   trapezoid, trapezoid/c_litres]
        scales = np.asarray(V005_EWM_TIMESCALES_SECONDS, dtype=np.float64)
        histories = np.empty((*u.shape, len(scales)), dtype=np.float32)
        state = np.broadcast_to(u[:, :1], (len(u), len(scales))).copy()
        histories[:, 0, :] = state
        # One loop over 80 timesteps; all breaths and timescales run together.
        # Irregular-time EWMA includes the control observed at the current step.
        for step in range(1, STEPS):
            alpha = np.exp(-dt[:, step, None]/scales[None, :])
            state = alpha*state + (1.-alpha)*u[:, step, None]
            histories[:, step, :] = state
        fixed = np.stack(columns, axis=-1).astype(np.float32)
        return np.concatenate((fixed, histories), axis=-1)


    def make_sequence_features(frame, feature_mode='causal20'):
        """Return (X[breaths,80,F] float32, names, breath_ids), in input order.

        causal20: exact legacy features; offline: legacy + known future/context;
        dynamics: legacy + causal dynamics proxies; combined: all three blocks.
        No scaling, fitting, target reads, IDs as features, or frame mutation occur.
        """
        names = feature_names_v005(feature_mode)
        breath_ids = _validate_feature_frame_v005(frame)
        base_flat, base_names, old_ids = features_causal20(frame)
        if base_names != list(V005_BASE_FEATURE_NAMES) or not np.array_equal(breath_ids, old_ids):
            raise ValueError('Legacy feature schema changed')
        base = base_flat.reshape(-1, STEPS, len(base_names))
        if feature_mode == 'causal20':
            return base, names, breath_ids.copy()
        result = np.empty((len(breath_ids), STEPS, len(names)), dtype=np.float32)
        result[..., :len(base_names)] = base
        position = len(base_names)
        if feature_mode in ('offline', 'combined'):
            block = _offline_feature_block_v005(frame)
            result[..., position:position+block.shape[-1]] = block
            position += block.shape[-1]
        if feature_mode in ('dynamics', 'combined'):
            block = _dynamics_feature_block_v005(frame)
            result[..., position:position+block.shape[-1]] = block
            position += block.shape[-1]
        if position != len(names) or not np.isfinite(result).all():
            raise ValueError('Invalid feature output')
        return result, names, breath_ids.copy()

In [ ]:
if RUN:
    # Imports and model definitions
    """Reusable v005 components. Defining these cells performs no data reads or fits."""
    import os
    os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
    import copy
    import hashlib
    import json
    import math
    import random
    import time
    import uuid
    from pathlib import Path
    import numpy as np
    import torch
    from torch import nn
    from torch.nn import functional as F
    from torch.utils.data import DataLoader, TensorDataset

    V5_STEPS = 80
    V5_R_VALUES = (5, 20, 50)
    V5_C_VALUES = (10, 20, 50)
    V5_MODEL_DEFAULTS = dict(family='gru', hidden=128, layers=2, dropout=.15,
        bidirectional=False, joint_embedding_dim=0, rc_heads=False, residual=False,
        head_width=32, widths=None, output_concat=False)
    V5_TRAIN_DEFAULTS = dict(lr=.0015, batch_size=256, max_epochs=150, patience=0,
        min_delta=1e-6, max_seconds=None, min_lr=1e-5, scheduler='plateau',
        scheduler_factor=.5, scheduler_patience=6, scheduler_threshold=1e-4,
        warmup_epochs=0, prediction_batch_size=512, gradient_clip=1.,
        optimizer='Adam', weight_decay=0., loss='l1', native_beta=.1,
        scaler_phase='all', residual_baseline=False, frozen_lr_schedule=None)


    def canonical_model_spec(model_spec):
        supplied = copy.deepcopy(dict(model_spec))
        unknown = set(supplied)-set(V5_MODEL_DEFAULTS)
        if unknown:
            raise ValueError(f'Unknown model fields: {sorted(unknown)}')
        spec = dict(V5_MODEL_DEFAULTS, **supplied)
        spec['family'] = str(spec['family']).lower()
        if spec['family'] not in ('gru', 'lstm'):
            raise ValueError('family must be gru or lstm')
        for key in ('hidden', 'layers', 'head_width'):
            spec[key] = int(spec[key])
            if spec[key] < 1:
                raise ValueError(f'{key} must be positive')
        spec['joint_embedding_dim'] = int(spec['joint_embedding_dim'])
        if spec['joint_embedding_dim'] < 0:
            raise ValueError('joint_embedding_dim must be nonnegative')
        spec['dropout'] = float(spec['dropout'])
        if not 0 <= spec['dropout'] < 1:
            raise ValueError('dropout must be in [0,1)')
        for key in ('bidirectional', 'rc_heads', 'residual', 'output_concat'):
            spec[key] = bool(spec[key])
        if spec['widths'] is not None:
            spec['widths'] = [int(v) for v in spec['widths']]
            if len(spec['widths']) != spec['layers'] or min(spec['widths']) < 1:
                raise ValueError('widths must contain one positive hidden width per layer')
        return spec


    def seed_sequence(seed):
        random.seed(int(seed))
        np.random.seed(int(seed))
        torch.manual_seed(int(seed))
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(int(seed))
        torch.set_num_threads(4)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.use_deterministic_algorithms(True)
        if hasattr(torch.backends.cuda.matmul, 'allow_tf32'):
            torch.backends.cuda.matmul.allow_tf32 = False
        if hasattr(torch.backends.cudnn, 'allow_tf32'):
            torch.backends.cudnn.allow_tf32 = False


    def _sequence_device(device):
        result = torch.device(device)
        if result.type == 'cuda' and not torch.cuda.is_available():
            raise RuntimeError('CUDA requested but unavailable')
        return result


    class V5PressureRNN(nn.Module):
        """Independent complete breaths; no hidden or cell state persists across calls.

        Transformed inputs contain F standardized numeric features, followed by an
        internal joint R/C code. Numeric R/C always remain among those F features.
        Bidirectional variants intentionally use future inputs within the same breath.
        """
        def __init__(self, model_spec, n_features):
            super().__init__()
            self.spec = spec = canonical_model_spec(model_spec)
            self.n_features = int(n_features)
            if self.n_features < 2:
                raise ValueError('At least the physical R and C channels are required')
            embedding = spec['joint_embedding_dim']
            self.joint_embedding = nn.Embedding(9, embedding) if embedding else None
            input_width = self.n_features + embedding
            directions = 2 if spec['bidirectional'] else 1
            rnn_class = nn.GRU if spec['family'] == 'gru' else nn.LSTM
            # Native multi-layer cuDNN recurrent dropout keeps opaque internal RNG state.
            # Single-layer recurrent modules plus explicit Torch dropout preserve the
            # plain stacked model while making every stochastic state serializable.
            self.explicit_stack = bool(spec['widths'] is not None or spec['residual'] or
                spec['output_concat'] or (spec['layers'] > 1 and spec['dropout'] > 0))
            if not self.explicit_stack:
                self.rnn = rnn_class(input_width, spec['hidden'], num_layers=spec['layers'],
                    batch_first=True, bidirectional=spec['bidirectional'],
                    dropout=spec['dropout'] if spec['layers'] > 1 else 0.)
                head_input = spec['hidden'] * directions
            else:
                widths = spec['widths'] or [spec['hidden']]*spec['layers']
                self.recurrent_layers = nn.ModuleList()
                self.skip_projections = nn.ModuleList()
                self.between_layers = nn.Dropout(spec['dropout'])
                outputs = []
                for width in widths:
                    output_width = width*directions
                    self.recurrent_layers.append(rnn_class(input_width, width, num_layers=1,
                        batch_first=True, bidirectional=spec['bidirectional']))
                    self.skip_projections.append(nn.Linear(input_width, output_width)
                        if spec['residual'] else nn.Identity())
                    outputs.append(output_width)
                    input_width = output_width
                head_input = sum(outputs) if spec['output_concat'] else outputs[-1]
            self.head = nn.Sequential(nn.Linear(head_input, spec['head_width']), nn.SiLU(),
                nn.Linear(spec['head_width'], 9 if spec['rc_heads'] else 1))

        def forward(self, x):
            if x.ndim != 3 or x.shape[-1] != self.n_features+1:
                raise ValueError('Expected standardized numeric features plus internal R/C code')
            joint = x[..., -1].long()
            sequence = x[..., :self.n_features]
            if self.joint_embedding is not None:
                sequence = torch.cat((sequence, self.joint_embedding(joint)), dim=-1)
            if not self.explicit_stack:
                sequence, _ = self.rnn(sequence)
            else:
                outputs = []
                for i, layer in enumerate(self.recurrent_layers):
                    previous = sequence
                    sequence, _ = layer(previous)
                    if self.spec['residual']:
                        sequence = sequence + self.skip_projections[i](previous)
                    outputs.append(sequence)
                    if i+1 < len(self.recurrent_layers):
                        sequence = self.between_layers(sequence)
                if self.spec['output_concat']:
                    sequence = torch.cat(outputs, dim=-1)
            prediction = self.head(sequence)
            if self.spec['rc_heads']:
                return prediction.gather(-1, joint.unsqueeze(-1)).squeeze(-1)
            return prediction.squeeze(-1)


    def build_model(model_spec, n_features):
        return V5PressureRNN(model_spec, n_features)


    def model_parameter_count(model_spec, n_features):
        return int(sum(p.numel() for p in build_model(model_spec, n_features).parameters()))

In [ ]:
if RUN:
    # Training-only preprocessing and native prediction
    def _raw_x(x):
        raw = np.asarray(x, dtype=np.float32)
        if raw.ndim != 3 or raw.shape[1] != V5_STEPS or raw.shape[2] < 2 or not len(raw):
            raise ValueError('Expected nonempty X[breaths,80,F], F >= 2')
        if not np.isfinite(raw).all():
            raise ValueError('Nonfinite feature')
        for column, values in ((0, V5_R_VALUES), (1, V5_C_VALUES)):
            if not np.isin(raw[..., column], values).all():
                raise ValueError('Unknown physical R/C value')
            if not np.all(raw[..., column] == raw[:, :1, column]):
                raise ValueError('R/C must be constant inside each breath')
        return raw


    def _training_arrays(x, y, u_out):
        raw = _raw_x(x)
        target, out = np.asarray(y, dtype=np.float32), np.asarray(u_out)
        if target.shape != raw.shape[:2] or out.shape != target.shape:
            raise ValueError('Feature/target/mask shape mismatch')
        if not np.isfinite(target).all() or not np.isin(out, [0, 1]).all() or not (out == 0).any():
            raise ValueError('Finite targets and a nonempty binary inspiration mask are required')
        return raw, target, out


    def _joint_codes(raw):
        r = np.searchsorted(np.asarray(V5_R_VALUES), raw[:, 0, 0])
        c = np.searchsorted(np.asarray(V5_C_VALUES), raw[:, 0, 1])
        return (3*r+c).astype(np.int64)


    def fit_preprocessor(x_train, y_train, u_out_train, model_spec, feature_names=None,
                         scaler_phase='all', residual_baseline=False):
        """All fitted statistics, including optional pressure profiles, use training only."""
        raw, y, out = _training_arrays(x_train, y_train, u_out_train)
        canonical_model_spec(model_spec)
        n_features = raw.shape[-1]
        names = list(feature_names) if feature_names is not None else None
        if names is not None and (len(names) != n_features or len(set(names)) != n_features):
            raise ValueError('feature_names must uniquely name every raw input channel')
        if scaler_phase not in ('all', 'inspiration'):
            raise ValueError('scaler_phase must be all or inspiration')
        mean = raw.mean(axis=(0, 1), dtype=np.float64).astype(np.float32)
        std = raw.std(axis=(0, 1), dtype=np.float64).astype(np.float32)
        phase_columns = []
        mask = out == 0
        if scaler_phase == 'inspiration':
            if names is None:
                raise ValueError('Inspiration-only scaling requires explicit feature_names')
            excluded = {'R', 'C', 'u_out', 'time_step', 'step'}
            phase_columns = [j for j, name in enumerate(names) if j >= 2 and name not in excluded]
            scored_x = raw[mask]
            phase_mean = scored_x.mean(axis=0, dtype=np.float64)
            phase_std = scored_x.std(axis=0, dtype=np.float64)
            mean[phase_columns] = phase_mean[phase_columns]
            std[phase_columns] = phase_std[phase_columns]
        std[std < 1e-6] = 1.
        baseline = np.zeros((9, V5_STEPS), dtype=np.float32)
        baseline_counts = np.zeros_like(baseline, dtype=np.int64)
        codes = _joint_codes(raw)
        if residual_baseline:
            fallback = float(np.median(y[mask]))
            pooled = np.array([np.median(y[:, t][mask[:, t]]) if mask[:, t].any()
                               else fallback for t in range(V5_STEPS)], dtype=np.float32)
            baseline[:] = pooled
            for group in range(9):
                group_mask = mask & (codes[:, None] == group)
                for step in range(V5_STEPS):
                    selected = group_mask[:, step]
                    baseline_counts[group, step] = int(selected.sum())
                    if selected.any():
                        baseline[group, step] = np.median(y[selected, step])
        residual = y-baseline[codes]
        scored = residual[mask]
        q25, q75 = np.quantile(scored, [.25, .75])
        return dict(input_shape=(V5_STEPS, n_features), feature_names=names,
            x_mean=mean, x_std=std, scaler_phase=scaler_phase,
            inspiration_scale_columns=phase_columns, target_center=float(np.median(scored)),
            target_scale=max(float(q75-q25), 1.), residual_baseline=bool(residual_baseline),
            baseline_profile=baseline, baseline_counts=baseline_counts,
            category_mappings=dict(R=list(V5_R_VALUES), C=list(V5_C_VALUES)),
            fitted_training_breaths=int(len(raw)), fitted_inspiratory_points=int(mask.sum()))


    def _validated_preprocessor(preprocess):
        prep = copy.deepcopy(preprocess)
        shape = tuple(prep['input_shape'])
        if len(shape) != 2 or shape[0] != V5_STEPS or shape[1] < 2:
            raise ValueError('Invalid checkpoint input shape')
        prep['input_shape'] = shape
        for key in ('x_mean', 'x_std'):
            prep[key] = np.array(prep[key], dtype=np.float32, copy=True)
            if prep[key].shape != (shape[1],) or not np.isfinite(prep[key]).all():
                raise ValueError('Invalid checkpoint input scaler')
        if (prep['x_std'] <= 0).any() or not np.isfinite(prep['target_center']):
            raise ValueError('Invalid input scale or target center')
        if not np.isfinite(prep['target_scale']) or prep['target_scale'] <= 0:
            raise ValueError('Invalid target scale')
        prep['baseline_profile'] = np.array(prep['baseline_profile'], dtype=np.float32, copy=True)
        if prep['baseline_profile'].shape != (9, V5_STEPS) or not np.isfinite(prep['baseline_profile']).all():
            raise ValueError('Invalid baseline profile')
        expected = dict(R=list(V5_R_VALUES), C=list(V5_C_VALUES))
        if prep['category_mappings'] != expected:
            raise ValueError('Checkpoint category mapping differs')
        return prep


    def transform_inputs(x, preprocess):
        raw, prep = _raw_x(x), _validated_preprocessor(preprocess)
        if raw.shape[1:] != prep['input_shape']:
            raise ValueError('Raw input shape differs from fitted preprocessor')
        standardized = (raw-prep['x_mean'])/prep['x_std']
        codes = np.broadcast_to(_joint_codes(raw)[:, None, None], (*raw.shape[:2], 1))
        transformed = np.ascontiguousarray(np.concatenate((standardized, codes), axis=-1), dtype=np.float32)
        if not np.isfinite(transformed).all():
            raise ValueError('Nonfinite standardized input')
        return transformed


    def _scaled_targets(raw, y, prep):
        return np.ascontiguousarray((y-prep['baseline_profile'][_joint_codes(raw)]
            -prep['target_center'])/prep['target_scale'], dtype=np.float32)


    def inspiratory_mae(y, prediction, u_out):
        y, prediction, out = np.asarray(y), np.asarray(prediction), np.asarray(u_out)
        if y.shape != prediction.shape or out.shape != y.shape or not (out == 0).any():
            raise ValueError('Metric shape or mask mismatch')
        if not np.isfinite(y).all() or not np.isfinite(prediction).all():
            raise ValueError('Nonfinite metric input')
        return float(np.abs(prediction.astype(np.float64)-y.astype(np.float64))[out == 0].mean())


    @torch.no_grad()
    def _predict_native(model, transformed, prep, device, batch_size):
        model.eval()
        output = np.empty(transformed.shape[:2], dtype=np.float32)
        for start in range(0, len(transformed), int(batch_size)):
            batch = torch.from_numpy(transformed[start:start+batch_size]).to(device)
            output[start:start+batch_size] = model(batch).cpu().numpy()
        codes = transformed[:, 0, -1].astype(np.int64)
        output = output*prep['target_scale']+prep['target_center']+prep['baseline_profile'][codes]
        if not np.isfinite(output).all():
            raise RuntimeError('Nonfinite native prediction')
        return output.astype(np.float32, copy=False)


    def load_checkpoint(checkpoint_path, device='cpu'):
        """Load trusted local v005 checkpoints. Older checkpoints need the old adapter."""
        payload = torch.load(Path(checkpoint_path), map_location='cpu', weights_only=False)
        if payload.get('format_version') != 5:
            raise ValueError('Expected a v005 checkpoint; use the matching older adapter')
        prep = _validated_preprocessor(payload['preprocess'])
        model = build_model(payload['model_spec'], prep['input_shape'][1]).to(_sequence_device(device))
        model.load_state_dict(payload['state_dict'], strict=True)
        model.eval()
        return model, prep, payload


    def predict_checkpoint(checkpoint_path, x, device='cpu', batch_size=512):
        model, prep, _ = load_checkpoint(checkpoint_path, device)
        return _predict_native(model, transform_inputs(x, prep), prep, _sequence_device(device), batch_size)

In [ ]:
if RUN:
    # Schedules, checkpointing, and reproducible epoch primitives
    def _train_configuration(train_cfg):
        supplied = copy.deepcopy(dict(train_cfg))
        unknown = set(supplied)-set(V5_TRAIN_DEFAULTS)
        if unknown:
            raise ValueError(f'Unknown train fields: {sorted(unknown)}')
        cfg = dict(V5_TRAIN_DEFAULTS, **supplied)
        for key in ('batch_size', 'max_epochs', 'prediction_batch_size'):
            cfg[key] = int(cfg[key])
            if cfg[key] < 1:
                raise ValueError(f'{key} must be positive')
        cfg['patience'] = 0 if cfg['patience'] is None else int(cfg['patience'])
        cfg['scheduler_patience'], cfg['warmup_epochs'] = int(cfg['scheduler_patience']), int(cfg['warmup_epochs'])
        if min(cfg['patience'], cfg['scheduler_patience'], cfg['warmup_epochs']) < 0:
            raise ValueError('Patience and warmup counts must be nonnegative')
        if not (0 < cfg['lr'] and 0 <= cfg['min_lr'] <= cfg['lr'] and cfg['gradient_clip'] > 0):
            raise ValueError('Invalid learning-rate or clipping configuration')
        if cfg['max_seconds'] is not None and cfg['max_seconds'] <= 0:
            raise ValueError('max_seconds must be positive or None')
        if cfg['min_delta'] < 0 or cfg['weight_decay'] < 0 or not 0 < cfg['scheduler_factor'] < 1:
            raise ValueError('Invalid regularization or selection configuration')
        if cfg['scheduler_threshold'] < 0:
            raise ValueError('scheduler_threshold must be nonnegative')
        if cfg['scheduler'] not in ('plateau', 'cosine', 'constant', 'replay', 'fixed'):
            raise ValueError('scheduler must be plateau, cosine, constant, or replay/fixed')
        if cfg['warmup_epochs'] >= cfg['max_epochs']:
            raise ValueError('Warmup must end before the total epoch budget')
        if cfg['scheduler'] != 'cosine' and cfg['warmup_epochs']:
            raise ValueError('warmup_epochs is supported by the cosine schedule')
        if cfg['scheduler'] in ('replay', 'fixed'):
            if cfg['frozen_lr_schedule'] is None:
                raise ValueError('Replay scheduling requires frozen_lr_schedule')
            rates = [float(v) for v in cfg['frozen_lr_schedule']]
            if len(rates) != cfg['max_epochs'] or not all(np.isfinite(v) and v >= 0 for v in rates):
                raise ValueError('Frozen rates must contain max_epochs nonnegative finite values')
            cfg['frozen_lr_schedule'] = rates
        elif cfg['frozen_lr_schedule'] is not None:
            raise ValueError('frozen_lr_schedule applies only to replay/fixed scheduling')
        if cfg['optimizer'] not in ('Adam', 'AdamW'):
            raise ValueError('optimizer must be Adam or AdamW')
        if cfg['loss'] not in ('l1', 'smooth_l1') or cfg['native_beta'] <= 0:
            raise ValueError('loss must be l1 or smooth_l1 with native_beta > 0')
        if cfg['scaler_phase'] not in ('all', 'inspiration'):
            raise ValueError('Invalid scaler phase')
        for key, value in cfg.items():
            if isinstance(value, (float, np.floating)) and not np.isfinite(value):
                raise ValueError(f'Nonfinite configuration value: {key}')
        return cfg


    def deterministic_lr_schedule(train_cfg):
        """Return upfront LRs for cosine/constant/replay; plateau requires A feedback."""
        cfg = _train_configuration(train_cfg)
        if cfg['scheduler'] == 'plateau':
            return None
        if cfg['scheduler'] in ('replay', 'fixed'):
            return list(cfg['frozen_lr_schedule'])
        if cfg['scheduler'] == 'constant':
            return [float(cfg['lr'])]*cfg['max_epochs']
        warmup, total = cfg['warmup_epochs'], cfg['max_epochs']
        rates = []
        for epoch in range(1, total+1):
            if warmup and epoch <= warmup:
                rate = cfg['lr']*epoch/warmup
            else:
                denominator = max(total-warmup-1, 1)
                progress = (epoch-warmup-1)/denominator
                rate = cfg['min_lr']+.5*(cfg['lr']-cfg['min_lr'])*(1+math.cos(math.pi*progress))
            rates.append(float(rate))
        return rates


    def _cpu_copy(value):
        if torch.is_tensor(value):
            return value.detach().cpu().clone()
        if isinstance(value, dict):
            return {k: _cpu_copy(v) for k, v in value.items()}
        if isinstance(value, list):
            return [_cpu_copy(v) for v in value]
        if isinstance(value, tuple):
            return tuple(_cpu_copy(v) for v in value)
        return copy.deepcopy(value)


    def _rng_state(generator):
        return dict(python=random.getstate(), numpy=np.random.get_state(), torch=torch.get_rng_state(),
            cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
            loader=generator.get_state())


    def _restore_rng(state, generator):
        random.setstate(state['python'])
        np.random.set_state(state['numpy'])
        torch.set_rng_state(state['torch'])
        if state['cuda'] is not None and torch.cuda.is_available():
            if len(state['cuda']) != torch.cuda.device_count():
                raise ValueError('CUDA device count changed across exact resumption')
            torch.cuda.set_rng_state_all(state['cuda'])
        generator.set_state(state['loader'])


    def _atomic_torch_save(payload, path):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_name(path.name+'.tmp-'+uuid.uuid4().hex)
        try:
            torch.save(payload, temporary)
            os.replace(temporary, path)
        finally:
            if temporary.exists():
                temporary.unlink()


    def _file_sha256(path):
        result = hashlib.sha256()
        with Path(path).open('rb') as stream:
            for block in iter(lambda: stream.read(1024*1024), b''):
                result.update(block)
        return result.hexdigest()


    def _array_sha256(values):
        array = np.ascontiguousarray(values)
        result = hashlib.sha256(str((array.shape, array.dtype.str)).encode())
        view = memoryview(array).cast('B')
        for start in range(0, len(view), 1024*1024):
            result.update(view[start:start+1024*1024])
        return result.hexdigest()


    def _json_safe(value):
        if isinstance(value, dict):
            return {k: _json_safe(v) for k, v in value.items()}
        if isinstance(value, (list, tuple)):
            return [_json_safe(v) for v in value]
        if isinstance(value, np.ndarray):
            return value.tolist()
        if isinstance(value, np.generic):
            return value.item()
        return value


    def _epoch_event(path, event):
        if path is not None:
            path = Path(path)
            path.parent.mkdir(parents=True, exist_ok=True)
            with path.open('a') as stream:
                stream.write(json.dumps(_json_safe(event), allow_nan=False)+'\n')


    def _last_path(checkpoint_path):
        path = Path(checkpoint_path)
        return path.with_name(path.stem+'.last'+path.suffix)


    def _fingerprint(raw, y, out, spec, cfg, seed, feature_names, mode, extra=None):
        return dict(mode=mode, model_spec=spec, train_cfg=cfg, seed=int(seed),
            feature_names=None if feature_names is None else list(feature_names),
            x_sha256=_array_sha256(raw), y_sha256=_array_sha256(y),
            out_sha256=_array_sha256(out), extra=extra)


    def _optimizer(model, cfg):
        optimizer_class = torch.optim.AdamW if cfg['optimizer'] == 'AdamW' else torch.optim.Adam
        return optimizer_class(model.parameters(), lr=float(cfg['lr']),
            betas=(.9, .999), eps=1e-8, weight_decay=float(cfg['weight_decay']))


    def _loader(raw, y, out, prep, cfg, seed, device):
        generator = torch.Generator().manual_seed(int(seed))
        transformed = transform_inputs(raw, prep)
        data = TensorDataset(torch.from_numpy(transformed), torch.from_numpy(_scaled_targets(raw, y, prep)),
            torch.from_numpy(np.ascontiguousarray(out == 0)))
        loader = DataLoader(data, batch_size=cfg['batch_size'], shuffle=True, num_workers=0,
            pin_memory=device.type == 'cuda', generator=generator)
        return loader, generator, transformed


    def _training_epoch(model, loader, optimizer, cfg, prep, device):
        model.train()
        total_error, total_count, optimizer_steps = 0., 0, 0
        for xb, yb, mask in loader:
            xb, yb, mask = (v.to(device, non_blocking=device.type == 'cuda') for v in (xb, yb, mask))
            if not bool(mask.any()):
                continue
            optimizer.zero_grad(set_to_none=True)
            difference = (model(xb)-yb).masked_select(mask)
            if cfg['loss'] == 'l1':
                loss = difference.abs().mean()
            else:
                beta = float(cfg['native_beta'])/prep['target_scale']
                loss = F.smooth_l1_loss(difference, torch.zeros_like(difference), beta=beta)
            if not torch.isfinite(loss):
                raise RuntimeError('Nonfinite training loss')
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), float(cfg['gradient_clip']))
            optimizer.step()
            optimizer_steps += 1
            total_error += float(difference.detach().abs().sum())*prep['target_scale']
            total_count += difference.numel()
        if not total_count:
            raise ValueError('Training epoch contained no scored points')
        return float(total_error/total_count), total_count, optimizer_steps


    def _checkpoint_payload(model, prep, spec, cfg, seed, fingerprint, optimizer, scheduler,
                            generator, epoch, history, completed=False, termination=None):
        return dict(format_version=5, model_spec=copy.deepcopy(spec), preprocess=_cpu_copy(prep),
            state_dict=_cpu_copy(model.state_dict()), optimizer_state=_cpu_copy(optimizer.state_dict()),
            scheduler_state=None if scheduler is None else copy.deepcopy(scheduler.state_dict()),
            rng_state=_cpu_copy(_rng_state(generator)), seed=int(seed), config=copy.deepcopy(cfg),
            fingerprint=copy.deepcopy(fingerprint), epoch=int(epoch), epochs=copy.deepcopy(history),
            optimizer_steps=sum(r['optimizer_steps'] for r in history),
            parameter_count=int(sum(p.numel() for p in model.parameters())),
            completed=bool(completed), termination=termination)


    def _prepare_run(raw, y, out, spec, cfg, seed, feature_names, checkpoint_path, resume,
                     fingerprint, device):
        checkpoint_path, last = Path(checkpoint_path), _last_path(checkpoint_path)
        if not resume and (checkpoint_path.exists() or last.exists()):
            raise FileExistsError('Use a fresh path or explicitly resume an existing stage')
        seed_sequence(seed)
        if resume:
            if not last.is_file():
                raise FileNotFoundError(f'Missing last-state checkpoint: {last}')
            saved = torch.load(last, map_location='cpu', weights_only=False)
            if saved.get('format_version') != 5 or saved['fingerprint'] != fingerprint:
                raise ValueError('Resume rejected: stage data, recipe, seed, or budget changed')
            prep = _validated_preprocessor(saved['preprocess'])
        else:
            saved = None
            prep = fit_preprocessor(raw, y, out, spec, feature_names,
                scaler_phase=cfg['scaler_phase'], residual_baseline=cfg['residual_baseline'])
        model = build_model(spec, raw.shape[-1]).to(device)
        optimizer = _optimizer(model, cfg)
        loader, generator, transformed = _loader(raw, y, out, prep, cfg, seed, device)
        return model, optimizer, loader, generator, transformed, prep, saved

In [ ]:
if RUN:
    # A validation fitting with best state and exact boundary resumption
    def fit_validate(x_train, y_train, u_out_train, x_val, y_val, u_out_val,
                     model_spec, train_cfg, seed, checkpoint_path, epoch_log_path=None,
                     feature_names=None, train_diagnostic_breaths=2048, device='cuda',
                     resume=False, on_epoch=None):
        """Select on A only. Callers must supply disjoint whole-breath groups.

        `patience=0` or None disables early stopping. A time cap stops after the current
        epoch and returns completed=False; resume continues from the separate .last
        checkpoint, preserving Adam moments, scheduler, all RNGs and shuffle generator.
        The public checkpoint always contains the best A model; .last also retains the
        complete best snapshot so an interruption between file writes is recoverable.
        """
        began = time.monotonic()
        device, spec, cfg = _sequence_device(device), canonical_model_spec(model_spec), _train_configuration(train_cfg)
        raw, y, out = _training_arrays(x_train, y_train, u_out_train)
        val_raw, val_y, val_out = _training_arrays(x_val, y_val, u_out_val)
        if val_raw.shape[1:] != raw.shape[1:]:
            raise ValueError('Training and validation feature dimensions differ')
        if int(train_diagnostic_breaths) < 1:
            raise ValueError('train_diagnostic_breaths must be positive')
        extra = dict(val_x_sha256=_array_sha256(val_raw), val_y_sha256=_array_sha256(val_y),
            val_out_sha256=_array_sha256(val_out), train_diagnostic_breaths=int(train_diagnostic_breaths))
        fingerprint = _fingerprint(raw, y, out, spec, cfg, seed, feature_names, 'validation', extra)
        model, optimizer, loader, generator, train_z, prep, saved = _prepare_run(
            raw, y, out, spec, cfg, seed, feature_names, checkpoint_path, resume, fingerprint, device)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
            factor=cfg['scheduler_factor'], patience=cfg['scheduler_patience'],
            threshold=cfg['scheduler_threshold'], threshold_mode='rel', min_lr=cfg['min_lr']) \
            if cfg['scheduler'] == 'plateau' else None
        upfront_rates = deterministic_lr_schedule(cfg)
        val_z = transform_inputs(val_raw, prep)
        diagnostic_ids = np.sort(np.random.default_rng(20261110).choice(len(raw),
            size=min(int(train_diagnostic_breaths), len(raw)), replace=False))
        history, best_payload, bad_epochs, first_epoch, elapsed_before = [], None, 0, 1, 0.
        termination, completed = 'epoch_cap', False
        if saved is not None:
            model.load_state_dict(saved['state_dict'], strict=True)
            optimizer.load_state_dict(saved['optimizer_state'])
            if scheduler is not None:
                scheduler.load_state_dict(saved['scheduler_state'])
            _restore_rng(saved['rng_state'], generator)
            history, best_payload = copy.deepcopy(saved['epochs']), saved.get('best_payload')
            bad_epochs, first_epoch = saved.get('bad_epochs', 0), saved['epoch']+1
            elapsed_before = float(saved.get('elapsed_seconds', 0.))
            completed, termination = saved['completed'], saved['termination']
        else:
            initial = _checkpoint_payload(model, prep, spec, cfg, seed, fingerprint, optimizer,
                scheduler, generator, 0, [], completed=False, termination='initialized')
            initial.update(best_payload=None, bad_epochs=0, elapsed_seconds=0.)
            _atomic_torch_save(initial, _last_path(checkpoint_path))
        _epoch_event(epoch_log_path, dict(event='resume' if resume else 'start', first_epoch=first_epoch,
            model_spec=spec, train_cfg=cfg, deterministic_lr_schedule=upfront_rates,
            optimizer=cfg['optimizer'], weight_decay=cfg['weight_decay'],
            loss=cfg['loss'], scaled_beta=cfg['native_beta']/prep['target_scale'] if cfg['loss']=='smooth_l1' else None))
        if not completed:
            for epoch in range(first_epoch, cfg['max_epochs']+1):
                epoch_began = time.monotonic()
                if upfront_rates is not None:
                    for group in optimizer.param_groups:
                        group['lr'] = upfront_rates[epoch-1]
                lr_used = float(optimizer.param_groups[0]['lr'])
                online_mae, count, optimizer_steps = _training_epoch(model, loader, optimizer, cfg, prep, device)
                validation_mae = inspiratory_mae(val_y,
                    _predict_native(model, val_z, prep, device, cfg['prediction_batch_size']), val_out)
                train_eval_mae = inspiratory_mae(y[diagnostic_ids],
                    _predict_native(model, train_z[diagnostic_ids], prep, device, cfg['prediction_batch_size']),
                    out[diagnostic_ids])
                best_mae = float('inf') if best_payload is None else best_payload['best_val_mae']
                improved = validation_mae < best_mae-float(cfg['min_delta'])
                bad_epochs = 0 if improved else bad_epochs+1
                if scheduler is not None:
                    scheduler.step(validation_mae)
                if upfront_rates is not None:
                    lr_next = upfront_rates[epoch] if epoch < len(upfront_rates) else None
                else:
                    lr_next = float(optimizer.param_groups[0]['lr'])
                elapsed = elapsed_before+time.monotonic()-began
                record = dict(event='epoch', epoch=epoch, lr_used=lr_used, lr_next=lr_next,
                    training_online_inspiratory_mae=online_mae, training_eval_inspiratory_mae=train_eval_mae,
                    training_diagnostic_breaths=len(diagnostic_ids), training_scored_points=count,
                    optimizer_steps=optimizer_steps,
                    cumulative_optimizer_steps=sum(r['optimizer_steps'] for r in history)+optimizer_steps,
                    validation_inspiratory_mae=validation_mae,
                    best_val_mae=validation_mae if improved else best_mae,
                    best_epoch=epoch if improved else best_payload['best_epoch'],
                    epoch_seconds=time.monotonic()-epoch_began, elapsed_seconds=elapsed)
                history.append(record)
                completed, termination = epoch == cfg['max_epochs'], 'epoch_cap'
                if cfg['patience'] and bad_epochs >= cfg['patience']:
                    completed, termination = True, 'early_stopping'
                if not completed and cfg['max_seconds'] is not None and time.monotonic()-began >= cfg['max_seconds']:
                    termination = 'time_cap'
                current = _checkpoint_payload(model, prep, spec, cfg, seed, fingerprint, optimizer,
                    scheduler, generator, epoch, history, completed, termination)
                current.update(bad_epochs=bad_epochs, elapsed_seconds=elapsed)
                if improved:
                    best_payload = copy.deepcopy(current)
                    best_payload.update(best_epoch=epoch, best_val_mae=validation_mae,
                        training_eval_mae=train_eval_mae,
                        selected_lr_schedule=[r['lr_used'] for r in history], validation_used=True)
                current['best_payload'] = best_payload
                # Last is authoritative for resumption, including its best snapshot.
                _atomic_torch_save(current, _last_path(checkpoint_path))
                if improved:
                    _atomic_torch_save(best_payload, checkpoint_path)
                _epoch_event(epoch_log_path, record)
                if on_epoch is not None:
                    on_epoch(copy.deepcopy(record))
                if completed or termination == 'time_cap':
                    break
        if best_payload is None:
            raise RuntimeError('No validation epoch has completed')
        model.load_state_dict(best_payload['state_dict'], strict=True)
        restored = inspiratory_mae(val_y,
            _predict_native(model, val_z, prep, device, cfg['prediction_batch_size']), val_out)
        if not np.isclose(restored, best_payload['best_val_mae'], rtol=1e-6, atol=1e-6):
            raise RuntimeError('Restored checkpoint metric differs')
        best_payload.update(executed_epochs=len(history), executed_lr_schedule=[r['lr_used'] for r in history],
            completed=bool(completed), termination=termination,
            stage_completed=bool(completed), stage_termination=termination)
        _atomic_torch_save(best_payload, checkpoint_path)
        report = dict(model_spec=spec, train_cfg=cfg, seed=int(seed), validation_mae=restored,
            best_epoch=best_payload['best_epoch'], selected_lr_schedule=best_payload['selected_lr_schedule'],
            deterministic_lr_schedule=upfront_rates, epochs=history, executed_epochs=len(history),
            executed_lr_schedule=[r['lr_used'] for r in history], completed=bool(completed), termination=termination,
            optimizer_steps=sum(r['optimizer_steps'] for r in history),
            training_eval_mae=best_payload['training_eval_mae'], training_diagnostic_breaths=len(diagnostic_ids),
            training_breaths=len(raw), validation_breaths=len(val_raw),
            scored_validation_points=int((val_out == 0).sum()),
            elapsed_seconds=elapsed_before+time.monotonic()-began,
            parameter_count=best_payload['parameter_count'], checkpoint_path=str(checkpoint_path),
            checkpoint_sha256=_file_sha256(checkpoint_path), last_checkpoint_path=str(_last_path(checkpoint_path)),
            preprocessing=_json_safe(prep), validation_used=True, best_checkpoint_restored=True)
        _epoch_event(epoch_log_path, dict(event='finished', completed=completed, termination=termination,
            best_epoch=report['best_epoch'], validation_mae=restored, checkpoint_sha256=report['checkpoint_sha256']))
        return _json_safe(report)

In [ ]:
if RUN:
    # Frozen A-schedule fitting for B or all labeled breaths
    def fit_fixed(x_train, y_train, u_out_train, model_spec, train_cfg, seed, lr_schedule,
                  checkpoint_path, epoch_log_path=None, feature_names=None, device='cuda',
                  resume=False, on_epoch=None):
        """Replay exactly one supplied LR per full epoch; accepts no validation arrays.

        Fits fresh training-only scalers/profiles. Saves each completed epoch and all
        optimizer/RNG states. No scheduler or early stopping consults B/all-data labels.
        A time cap returns completed=False and can be resumed at the next epoch.
        """
        began = time.monotonic()
        device, spec, cfg = _sequence_device(device), canonical_model_spec(model_spec), _train_configuration(train_cfg)
        raw, y, out = _training_arrays(x_train, y_train, u_out_train)
        rates = [float(rate) for rate in lr_schedule]
        if not rates or not all(np.isfinite(rate) and rate >= 0 for rate in rates):
            raise ValueError('A nonempty frozen sequence of nonnegative finite LRs is required')
        fingerprint = _fingerprint(raw, y, out, spec, cfg, seed, feature_names, 'fixed',
            dict(lr_schedule=rates))
        model, optimizer, loader, generator, _, prep, saved = _prepare_run(
            raw, y, out, spec, cfg, seed, feature_names, checkpoint_path, resume, fingerprint, device)
        history, first_epoch, elapsed_before = [], 1, 0.
        completed, termination = False, 'fixed_A_budget'
        if saved is not None:
            model.load_state_dict(saved['state_dict'], strict=True)
            optimizer.load_state_dict(saved['optimizer_state'])
            _restore_rng(saved['rng_state'], generator)
            history, first_epoch = copy.deepcopy(saved['epochs']), saved['epoch']+1
            elapsed_before = float(saved.get('elapsed_seconds', 0.))
            completed, termination = saved['completed'], saved['termination']
        else:
            initial = _checkpoint_payload(model, prep, spec, cfg, seed, fingerprint, optimizer,
                None, generator, 0, [], completed=False, termination='initialized')
            initial.update(elapsed_seconds=0., validation_used=False, selected_lr_schedule=rates)
            _atomic_torch_save(initial, _last_path(checkpoint_path))
        _epoch_event(epoch_log_path, dict(event='resume' if resume else 'start', first_epoch=first_epoch,
            model_spec=spec, train_cfg=cfg, frozen_lr_schedule=rates, validation_used=False))
        if not completed:
            for epoch in range(first_epoch, len(rates)+1):
                epoch_began = time.monotonic()
                for group in optimizer.param_groups:
                    group['lr'] = rates[epoch-1]
                online_mae, count, optimizer_steps = _training_epoch(model, loader, optimizer, cfg, prep, device)
                elapsed = elapsed_before+time.monotonic()-began
                record = dict(event='epoch', epoch=epoch, lr_used=rates[epoch-1],
                    lr_next=rates[epoch] if epoch < len(rates) else None,
                    training_online_inspiratory_mae=online_mae, training_scored_points=count,
                    optimizer_steps=optimizer_steps,
                    cumulative_optimizer_steps=sum(r['optimizer_steps'] for r in history)+optimizer_steps,
                    epoch_seconds=time.monotonic()-epoch_began, elapsed_seconds=elapsed, validation_used=False)
                history.append(record)
                completed, termination = epoch == len(rates), 'fixed_A_budget'
                if not completed and cfg['max_seconds'] is not None and time.monotonic()-began >= cfg['max_seconds']:
                    termination = 'time_cap'
                payload = _checkpoint_payload(model, prep, spec, cfg, seed, fingerprint, optimizer,
                    None, generator, epoch, history, completed, termination)
                payload.update(elapsed_seconds=elapsed, validation_used=False,
                    selected_lr_schedule=rates, completed_epochs=epoch)
                _atomic_torch_save(payload, _last_path(checkpoint_path))
                _atomic_torch_save(payload, checkpoint_path)
                _epoch_event(epoch_log_path, record)
                if on_epoch is not None:
                    on_epoch(copy.deepcopy(record))
                if completed or termination == 'time_cap':
                    break
        # A completed resume may have only the authoritative .last file after interruption.
        if saved is not None and first_epoch > len(rates):
            _atomic_torch_save(saved, checkpoint_path)
        report = dict(model_spec=spec, train_cfg=cfg, seed=int(seed), epochs=history,
            selected_lr_schedule=rates, executed_lr_schedule=[r['lr_used'] for r in history],
            executed_epochs=len(history), completed_epochs=len(history), completed=bool(completed),
            optimizer_steps=sum(r['optimizer_steps'] for r in history),
            termination=termination, validation_used=False, training_breaths=len(raw),
            parameter_count=int(sum(p.numel() for p in model.parameters())),
            elapsed_seconds=elapsed_before+time.monotonic()-began, checkpoint_path=str(checkpoint_path),
            checkpoint_sha256=_file_sha256(checkpoint_path), last_checkpoint_path=str(_last_path(checkpoint_path)),
            preprocessing=_json_safe(prep))
        _epoch_event(epoch_log_path, dict(event='finished', completed=completed, termination=termination,
            checkpoint_sha256=report['checkpoint_sha256']))
        return _json_safe(report)

## Initialization and immutable source/data/group design

In [ ]:
if RUN:
    import torch
    import lightgbm as lgb
    import platform
    from sklearn.model_selection import train_test_split
    OUT.mkdir(parents=True, exist_ok=True)
    RUN_LOCK = (OUT/'.run.lock').open('a')
    try:
        fcntl.flock(RUN_LOCK.fileno(), fcntl.LOCK_EX|fcntl.LOCK_NB)
    except BlockingIOError:
        raise RuntimeError('Another worker owns this confirmation round')
    quality = json.loads((PROJECT/'quality_audit_v002.json').read_text())
    train_sha = digest(DATA/'train.csv')
    assert train_sha == quality['files']['train.csv']['sha256']
    SEQ_DESIGN = json.loads((SEQ/'design.json').read_text())
    TREE_DESIGN = json.loads((TREE/'design.json').read_text())
    OLD_DESIGN = json.loads((OLD/'experiments/design.json').read_text())
    V3_DESIGN = json.loads((V3/'design.json').read_text())
    assert SEQ_DESIGN['train_sha256']==TREE_DESIGN['train_sha256']==OLD_DESIGN['train_sha256']==V3_DESIGN['train_sha256']==train_sha
    assert SEQ_DESIGN['source_code_sha256']==code_hash(PROJECT/'iterative_improvements_v005.ipynb')
    assert TREE_DESIGN['source_sha256']==code_hash(PROJECT/'tree_context_v005.ipynb')
    assert OLD_DESIGN['source_sha256']==code_hash(PROJECT/'sequence_refinement_v004b.ipynb')
    primary = json.loads((PROJECT/'iterative_improvements_v005.ipynb').read_text())
    def unwrap(cell):
        source = ''.join(cell['source'])
        return textwrap.dedent(source.split('\n',1)[1]).rstrip()+'\n' if source.startswith('if RUN:\n') else source.rstrip()+'\n'
    current = json.loads((PROJECT/'confirmation_final_v005.ipynb').read_text())
    embedded_sequence = [unwrap(c) for c in current['cells'] if 'v005-sequence-definition' in c.get('metadata',{}).get('tags',[])]
    embedded_features = [unwrap(c) for c in current['cells'] if 'v005-feature-definition' in c.get('metadata',{}).get('tags',[])]
    assert len(embedded_sequence)==5 and len(embedded_features)==3
    assert embedded_sequence==[unwrap(primary['cells'][i]) for i in [7,8,9,10,11]]
    assert embedded_features==[unwrap(primary['cells'][i]) for i in [3,4,5]]
    tree_nb=json.loads((PROJECT/'tree_context_v005.ipynb').read_text())
    assert embedded_features==[unwrap(c) for c in tree_nb['cells'] if 'v005-feature-definition' in c.get('metadata',{}).get('tags',[])]
    frame = pd.read_csv(DATA/'train.csv',dtype=DTYPES)
    BREATH_IDS = _validate_feature_frame_v005(frame)
    assert len(BREATH_IDS)==67905
    Y=frame.pressure.to_numpy(copy=True).reshape(-1,80)
    UOUT=frame.u_out.to_numpy(copy=True).reshape(-1,80)
    ROW_IDS=frame.id.to_numpy(copy=True).reshape(-1,80)
    RC=frame[['R','C']].to_numpy(copy=True)[::80]
    KNOWN_FRAME=frame[INPUT_COLUMNS].copy(); del frame; gc.collect()
    index=pd.Index(BREATH_IDS)
    with np.load(SEQ/'split_breath_ids.npz',allow_pickle=False) as splits:
        assert set(splits.files)=={'A_fit','A_validation','C_fit','C_validation'}
        GROUPS={key:np.sort(index.get_indexer(splits[key])) for key in splits.files}
        assert all((v>=0).all() and len(np.unique(v))==len(v) for v in GROUPS.values())
    assert len(GROUPS['A_fit'])==61114 and len(GROUPS['A_validation'])==6791
    assert len(GROUPS['C_fit'])==54323 and len(GROUPS['C_validation'])==6791
    assert np.array_equal(np.sort(np.r_[GROUPS['A_fit'],GROUPS['A_validation']]),np.arange(67905))
    assert np.array_equal(np.sort(np.r_[GROUPS['C_fit'],GROUPS['C_validation']]),GROUPS['A_fit'])
    for key in ['A_fit','C_fit','C_validation']:
        assert not np.intersect1d(GROUPS[key],GROUPS['A_validation']).size
    rc_code=RC[:,0].astype(int)*100+RC[:,1]
    expected_fit,expected_val=train_test_split(GROUPS['A_fit'],test_size=6791,
        random_state=20261117,stratify=rc_code[GROUPS['A_fit']])
    assert np.array_equal(np.sort(expected_fit),GROUPS['C_fit'])
    assert np.array_equal(np.sort(expected_val),GROUPS['C_validation'])
    GROUP_HASHES={key:array_hash(BREATH_IDS[value]) for key,value in GROUPS.items()}
    assert GROUP_HASHES==SEQ_DESIGN['group_hashes']
    for key in ['A_fit','A_validation']:
        assert GROUP_HASHES[key]==TREE_DESIGN['group_hashes'][key]
    assert GROUP_HASHES['A_fit']==V3_DESIGN['groups']['Afit']
    assert GROUP_HASHES['A_validation']==V3_DESIGN['groups']['Aval']
    ALL=np.arange(67905)
    A_IDS=GROUPS['A_validation']; A_MASK=UOUT[A_IDS]==0
    A_TARGET=Y[A_IDS][A_MASK].astype(np.float64); A_ROW_IDS=ROW_IDS[A_IDS][A_MASK]
    assert len(A_TARGET)==206098
    CACHE=OneModeFeatureCache()
    DESIGN=json.loads(json.dumps(dict(plan=PLAN,source_code_sha256=code_hash(PROJECT/'confirmation_final_v005.ipynb'),
        train_sha256=train_sha,sequence_design_sha256=canonical_hash(SEQ_DESIGN),
        tree_design_sha256=canonical_hash(TREE_DESIGN),old_design_sha256=canonical_hash(OLD_DESIGN),
        group_hashes=GROUP_HASHES,A_row_ids_sha256=array_hash(A_ROW_IDS),
        feature_names={mode:feature_names_v005(mode) for mode in V005_FEATURE_MODES},
        embedded_sequence_sha256=hashlib.sha256('\n'.join(embedded_sequence).encode()).hexdigest(),
        embedded_features_sha256=hashlib.sha256('\n'.join(embedded_features).encode()).hexdigest(),
        legacy_definition_sha256=hashlib.sha256('\n'.join(''.join(json.loads((PROJECT/'sequence_refinement_v004b.ipynb').read_text())['cells'][i]['source']) for i in [4,5,6,7]).encode()).hexdigest())))
    if (OUT/'design.json').exists():
        assert RESUME and json.loads((OUT/'design.json').read_text())==DESIGN,'Source/data/recipe changed; use a new round/version'
    else:
        preserve_partial([OUT/'A_row_ids.npy',OUT/'split_breath_ids.npz',OUT/'split_breath_ids.tmp.npz'],'initialization')
        save_array(OUT/'A_row_ids.npy',A_ROW_IDS)
        splittemp=OUT/'split_breath_ids.tmp.npz'
        np.savez_compressed(splittemp,**{k:BREATH_IDS[v] for k,v in GROUPS.items()})
        os.replace(splittemp,OUT/'split_breath_ids.npz')
        write_json(OUT/'design.json',DESIGN)
    assert np.array_equal(np.load(OUT/'A_row_ids.npy'),A_ROW_IDS)
    with np.load(OUT/'split_breath_ids.npz',allow_pickle=False) as stored:
        assert set(stored.files)==set(GROUPS)
        for key,value in GROUPS.items():assert np.array_equal(stored[key],BREATH_IDS[value])
    event('initialized',resume=RESUME,limit=LIMIT,source_sha256=DESIGN['source_code_sha256'],
        torch=torch.__version__,cuda=torch.version.cuda,platform=platform.platform(),
        gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        group_sizes={k:len(v) for k,v in GROUPS.items()})

## Strict A loaders and an explicit completed phase snapshot

In [ ]:
def verify_candidate_artifacts(candidate):
    for field, hash_field in [('source_record_file','source_record_sha256'),
                               ('checkpoint_path','checkpoint_sha256'),
                               ('prediction_file','prediction_file_sha256'),
                               ('row_id_file','row_id_file_sha256')]:
        if digest(resolve_artifact(candidate[field])) != candidate[hash_field]:
            raise ValueError('Locked A artifact changed: '+field)
    prediction=np.load(resolve_artifact(candidate['prediction_file']),allow_pickle=False)
    row_ids=np.load(resolve_artifact(candidate['row_id_file']),allow_pickle=False)
    assert array_hash(prediction)==candidate['prediction_array_sha256']
    score=aligned_mae(prediction,A_TARGET,row_ids,A_ROW_IDS)
    assert abs(score-candidate['A_mae'])<1e-10
    assert candidate['fit_group_sha256']==GROUP_HASHES['A_fit']
    assert candidate['feature_names']==feature_names_v005(candidate['feature_mode'])
    return np.asarray(prediction,dtype=np.float64)

def descriptor(name,kind,record,path,row_path,prediction_hash_kind='array',legacy_name=None):
    checkpoint=resolve_artifact(record['checkpoint_path'])
    prediction_path=resolve_artifact(record['prediction_file'])
    assert digest(checkpoint)==record['checkpoint_sha256']
    prediction=np.load(prediction_path,allow_pickle=False)
    expected=record['prediction_sha256']
    assert (array_hash(prediction) if prediction_hash_kind=='array' else digest(prediction_path))==expected
    score=aligned_mae(prediction,A_TARGET,np.load(row_path,allow_pickle=False),A_ROW_IDS)
    assert abs(score-record['validation_mae'])<1e-10
    mode=record.get('feature_mode','causal20')
    return dict(name=name,kind=kind,legacy_name=legacy_name,feature_mode=mode,
        feature_names=feature_names_v005(mode),recipe=copy.deepcopy(record['recipe']),
        seed=int(record['recipe'].get('seed',record.get('seed',20261102))),A_mae=score,
        selected_lr_schedule=copy.deepcopy(record.get('selected_lr_schedule')),
        best_epoch=record.get('best_epoch'),fixed_trees=record.get('best_iteration'),
        checkpoint_path=str(checkpoint),checkpoint_sha256=digest(checkpoint),
        prediction_file=str(prediction_path),prediction_file_sha256=digest(prediction_path),
        prediction_array_sha256=array_hash(prediction),row_id_file=str(row_path),
        row_id_file_sha256=digest(row_path),source_record_file=str(path),
        source_record_sha256=digest(path),fit_group_sha256=GROUP_HASHES['A_fit'])

def load_nn_A(path):
    r=json.loads(path.read_text()); recipe=r['recipe']
    assert r['phase'] in PHASES and PHASES.index(r['phase'])<=PHASES.index(A_PHASE)
    assert r['training_breaths']==61114 and r['fit_group_sha256']==GROUP_HASHES['A_fit']
    assert r['feature_mode']==recipe['feature_mode']
    assert r['feature_names']==feature_names_v005(r['feature_mode'])
    assert r['identity']==canonical_hash(dict(recipe=recipe,design=canonical_hash(SEQ_DESIGN),split='A'))
    report=json.loads(resolve_artifact(r['report_file']).read_text())
    assert report['completed'] and report['executed_epochs']==recipe['train_cfg']['max_epochs']
    assert r['executed_epochs']==report['executed_epochs']
    assert r['selected_lr_schedule']==report['selected_lr_schedule']
    assert r['executed_lr_schedule']==report['executed_lr_schedule']
    assert len(r['selected_lr_schedule'])==r['best_epoch']==report['best_epoch']
    assert len(r['executed_lr_schedule'])==r['executed_epochs']
    assert r['selected_lr_schedule']==r['executed_lr_schedule'][:r['best_epoch']]
    checkpoint=resolve_artifact(r['checkpoint_path'])
    assert digest(checkpoint)==r['checkpoint_sha256']==report['checkpoint_sha256']
    model,prep,payload=load_checkpoint(checkpoint,device='cpu'); del model
    assert payload['model_spec']==canonical_model_spec(recipe['model_spec'])
    assert payload['config']==_train_configuration(recipe['train_cfg'])
    assert prep['input_shape']==(80,len(r['feature_names'])) and prep['feature_names']==r['feature_names']
    assert prep['fitted_training_breaths']==61114
    assert prep['fitted_inspiratory_points']==int((UOUT[GROUPS['A_fit']]==0).sum())
    fp=payload['fingerprint']
    assert fp['mode']=='validation' and fp['seed']==r['seed']==recipe['seed']
    assert fp['feature_names']==r['feature_names']
    assert fp['y_sha256']==_array_sha256(Y[GROUPS['A_fit']])
    assert fp['out_sha256']==_array_sha256(UOUT[GROUPS['A_fit']])
    assert fp['extra']['val_y_sha256']==_array_sha256(Y[A_IDS])
    assert fp['extra']['val_out_sha256']==_array_sha256(UOUT[A_IDS])
    assert payload['best_epoch']==r['best_epoch'] and payload['stage_completed']
    result=descriptor('nn__'+r['name'],'v5_nn',r,path,SEQ/'A_validation_row_ids.npy')
    result['source_report_file']=str(resolve_artifact(r['report_file']))
    result['source_report_sha256']=digest(result['source_report_file'])
    return result

def load_tree_A(path):
    r=json.loads(path.read_text()); identity=r['identity']
    assert r['status']=='complete' and r['split']=='A' and r['training_breaths']==61114
    assert r['validation_breaths']==6791 and r['scored_points']==len(A_TARGET)
    assert r['identity_sha256']==canonical_hash(identity)
    assert r['design_sha256']==canonical_hash(TREE_DESIGN)==identity['design_sha256']
    assert r['source_sha256']==TREE_DESIGN['source_sha256']
    assert identity['fit_breath_sha256']==GROUP_HASHES['A_fit']
    assert identity['validation_breath_sha256']==GROUP_HASHES['A_validation']
    assert identity['feature_mode']==r['feature_mode']==r['recipe']['feature_mode']
    assert identity['input_columns']==r['input_columns']==feature_names_v005(r['feature_mode'])
    assert r['best_iteration']==r['recipe']['fixed_trees']==2500
    assert r['recipe']['fit_params']==TREE_DESIGN['plan']['fit_parameters']
    assert r['recipe']['fit_params']['n_jobs']==4 and r['recipe']['rc']=='numeric'
    assert digest(resolve_artifact(r['scored_row_id_file']))==r['scored_row_id_sha256']
    assert digest(resolve_artifact(r['rc_breakdown_file']))==r['rc_breakdown_sha256']
    booster=lgb.Booster(model_file=str(resolve_artifact(r['checkpoint_path'])))
    assert booster.num_trees()==2500 and booster.feature_name()==r['input_columns']; del booster
    return descriptor('tree__'+r['name'],'v5_tree',r,path,
        resolve_artifact(r['scored_row_id_file']),'file')

def load_legacy_A(name):
    path=OLD/'experiments'/('record_A_'+name+'.json')
    r=json.loads(path.read_text())
    assert r['training_breaths']==61114 and r['fit_group_sha256']==GROUP_HASHES['A_fit']
    if name=='warm_current':
        assert r['identity']==canonical_hash(dict(recipe=r['recipe'],design=canonical_hash(OLD_DESIGN),split='A'))
        assert r['best_epoch']==29 and r['selected_lr_schedule']==[1e-5]*29
        assert r['recipe']['base']=='gru_s20261102'
        kind='legacy_warm'
    else:
        old=r['old_metadata']; identity=old['identity']
        assert old['status']=='complete' and old['split']=='A'
        assert old['identity_sha256']==canonical_hash(identity)
        assert identity['design_sha256']==canonical_hash(V3_DESIGN)
        assert identity['fit_breath_sha256']==GROUP_HASHES['A_fit']
        assert identity['val_breath_sha256']==GROUP_HASHES['A_validation']
        assert old['input_columns']==feature_names_v005('causal20')
        if name=='gru_s20261102':
            assert r['best_epoch']==99 and len(r['selected_lr_schedule'])==99
            kind='legacy_base'
        else:
            assert name=='tree_511' and old['best_iteration']==2500
            assert old['definition']['params']['num_leaves']==511
            kind='legacy_tree'
    result=descriptor('legacy__'+name,kind,r,path,OLD/'experiments/A_row_ids.npy',legacy_name=name)
    if kind=='legacy_tree':
        params=copy.deepcopy(r['old_metadata']['definition']['params'])
        result['fixed_trees']=2500
        result['legacy_tree_parameters']=dict(params,objective='regression_l1',subsample=.9,
            subsample_freq=1,colsample_bytree=.9,n_jobs=4,random_state=20261102,
            deterministic=True,force_col_wise=True,verbosity=-1,device_type='cpu')
    return result

def require_A_snapshot():
    expected=['schedule_plateau_p1','schedule_plateau_p6','schedule_cosine_w5']
    files=['schedule_selection.json']
    if PHASES.index(A_PHASE)>=1:
        expected += ['features_offline','features_dynamics','features_combined'];files+=['feature_selection.json']
    if PHASES.index(A_PHASE)>=2:
        expected += ['capacity_uni_gru256','capacity_bigru128','capacity_bilstm128','capacity_deep_bilstm'];files+=['capacity_selection.json']
    if PHASES.index(A_PHASE)>=3:
        expected += ['refinement_'+n for n in ['joint_RC_embedding4','RC_heads','residual_profile','inspiration_scaler','dropout_005']]
        files+=['refinement_selection.json']
    if PHASES.index(A_PHASE)>=4:
        expected += ['seed_20261118','seed_20261119']
    for name in expected:
        assert (SEQ/('record_A_'+name+'.json')).is_file(),'A snapshot is not complete: '+name
    for name in files:
        assert (SEQ/name).is_file(),'A phase completion artifact missing: '+name
    for name in ['tree_causal20','tree_offline','tree_dynamics','tree_combined']:
        assert (TREE/('record_A_'+name+'.json')).is_file(),'Tree A snapshot is not complete: '+name
    for directory in [SEQ,TREE]:
        for filename in ['.run.lock','execution-worker.lock']:
            path=directory/filename
            if path.exists():
                with path.open('r') as stream:
                    try:fcntl.flock(stream.fileno(),fcntl.LOCK_EX|fcntl.LOCK_NB)
                    except BlockingIOError:raise RuntimeError('An A worker still owns '+str(path))
                    fcntl.flock(stream.fileno(),fcntl.LOCK_UN)
    return dict(phase=A_PHASE,expected_nn_names=expected,
        completion_artifacts={n:digest(SEQ/n) for n in files})

def lock_A():
    lock_path=OUT/'locked_before_C.json'
    if lock_path.exists():
        assert RESUME,'Existing frozen round requires PA3_RESUME_CONFIRM_V005=1'
        frozen=json.loads(lock_path.read_text())
        assert frozen['design_sha256']==canonical_hash(DESIGN)
        assert frozen['lock_sha256']==canonical_hash({k:v for k,v in frozen.items() if k!='lock_sha256'})
        for candidate in frozen['candidates'].values():
            verify_candidate_artifacts(candidate)
            if 'source_report_file' in candidate:
                assert digest(resolve_artifact(candidate['source_report_file']))==candidate['source_report_sha256']
        event('frozen_A_snapshot_reused',lock_sha256=frozen['lock_sha256'])
        return frozen
    snapshot=require_A_snapshot()
    candidates=[]
    for name in snapshot['expected_nn_names']:
        candidates.append(load_nn_A(SEQ/('record_A_'+name+'.json')))
    expected_trees=['tree_causal20','tree_offline','tree_dynamics','tree_combined']
    for name in expected_trees:
        candidates.append(load_tree_A(TREE/('record_A_'+name+'.json')))
    for name in ['gru_s20261102','warm_current','tree_511']:
        candidates.append(load_legacy_A(name))
    assert set(snapshot['expected_nn_names'])=={r['name'].removeprefix('nn__') for r in candidates if r['kind']=='v5_nn'}
    by_name={r['name']:r for r in candidates}
    assert len(by_name)==len(candidates)
    predictions={n:verify_candidate_artifacts(r) for n,r in by_name.items()}
    approaches,pairs=choose_A(candidates,predictions,A_TARGET,
        ['legacy__tree_511','legacy__warm_current'],[.25,.75])
    prior=json.loads((OLD/'experiments/validation_selection.json').read_text())
    assert prior['members']==['tree_511','warm_current'] and prior['weights']==[.25,.75]
    assert abs(approaches['reference']['A_mae']-prior['A_mae'])<1e-10
    required=set()
    for approach in approaches.values():required.update(approach['members'])
    # A C warm start must use its own C base, never any A or all-data weights.
    if 'legacy__warm_current' in required:required.add('legacy__gru_s20261102')
    frozen=dict(design_sha256=canonical_hash(DESIGN),A_snapshot=snapshot,candidates=by_name,
        approaches=approaches,C_required_members=sorted(required),A_pair_candidates=len(pairs),
        selected_using='A only; no C scores read',C_selection_universe=list(approaches),
        pair_weights_tuned_on_C=False,created_utc=datetime.now(timezone.utc).isoformat(),rule=PLAN['rule'])
    frozen['lock_sha256']=canonical_hash(frozen)
    write_json(lock_path,frozen)
    pd.DataFrame(pairs).sort_values('A_mae').to_csv(OUT/'A_pair_search.csv',index=False)
    pd.DataFrame([{k:r[k] for k in ['name','kind','feature_mode','A_mae','checkpoint_sha256']}
        for r in candidates]).sort_values('A_mae').to_csv(OUT/'A_snapshot.csv',index=False)
    event('A_SELECTION_LOCKED_BEFORE_C',lock_sha256=frozen['lock_sha256'],approaches=approaches,
        C_required_members=frozen['C_required_members'])
    return frozen

if RUN:
    FROZEN=lock_A()
    CANDIDATES=FROZEN['candidates']
    display(pd.DataFrame([dict(approach=k,**v) for k,v in FROZEN['approaches'].items()]))

## Isolated legacy adapter and fixed-budget C/all-data member fits

In [ ]:
LEGACY_NAMESPACE=None
FITTED_RECORDS={}
def legacy_namespace():
    global LEGACY_NAMESPACE
    if LEGACY_NAMESPACE is None:
        namespace={'RUN':True,'__name__':'v005_isolated_legacy_components'}
        notebook=json.loads((PROJECT/'sequence_refinement_v004b.ipynb').read_text())
        source='\n\n'.join(''.join(notebook['cells'][i]['source']) for i in [4,5,6,7])
        exec(compile(source,'sequence_refinement_v004b.ipynb:definitions','exec'),namespace)
        LEGACY_NAMESPACE=namespace
    return LEGACY_NAMESPACE

def train_features(mode):
    sequence,names,ids=CACHE.get(KNOWN_FRAME,mode,'approved_train')
    assert names==feature_names_v005(mode) and np.array_equal(ids,BREATH_IDS)
    assert sequence.shape==(67905,80,len(names))
    return sequence,names

def member_paths(stage,name,kind):
    directory=OUT/'C_models' if stage=='C' else FINAL
    directory.mkdir(parents=True,exist_ok=True)
    extension='.txt' if kind in ['v5_tree','legacy_tree'] else '.pt'
    return dict(record=directory/('record_'+name+'.json'),checkpoint=directory/(name+extension),
        report=directory/('report_'+name+'.json'),epochs=directory/(name+'_epochs.jsonl'),
        prediction=directory/(name+'_C_predictions.npy'))

def member_identity(candidate,stage):
    group='C_fit' if stage=='C' else 'all'
    return dict(lock_sha256=FROZEN['lock_sha256'],design_sha256=canonical_hash(DESIGN),
        member=candidate['name'],A_checkpoint_sha256=candidate['checkpoint_sha256'],
        A_source_record_sha256=candidate['source_record_sha256'],stage=stage,
        group=group,fit_group_sha256=GROUP_HASHES['C_fit'] if stage=='C' else array_hash(BREATH_IDS),
        recipe=candidate['recipe'],selected_lr_schedule=candidate['selected_lr_schedule'],
        fixed_trees=candidate['fixed_trees'],feature_names=candidate['feature_names'])

def verify_fitted(record,identity,stage):
    assert record['status']=='complete' and record['identity']==identity
    assert record['identity_sha256']==canonical_hash(identity)
    assert digest(resolve_artifact(record['checkpoint_path']))==record['checkpoint_sha256']
    assert record['fit_group_sha256']==identity['fit_group_sha256']
    assert record['training_breaths']==(54323 if stage=='C' else 67905)
    assert record['validation_feedback'] is False
    if 'report_file' in record:
        assert digest(resolve_artifact(record['report_file']))==record['report_sha256']
        report=json.loads(resolve_artifact(record['report_file']).read_text())
        assert report['training_breaths']==record['training_breaths']
        assert report['validation_used'] is False
        if record['kind']=='v5_nn':
            assert report['completed'] and report['executed_epochs']==len(record['selected_lr_schedule'])
            assert report['preprocessing']['fitted_training_breaths']==record['training_breaths']
            assert report['preprocessing']['fitted_inspiratory_points']==record['scored_training_rows']
    if stage=='C':
        p=np.load(resolve_artifact(record['prediction_file']),allow_pickle=False)
        assert digest(resolve_artifact(record['prediction_file']))==record['prediction_file_sha256']
        assert array_hash(p)==record['prediction_array_sha256']
        assert digest(resolve_artifact(record['row_id_file']))==record['row_id_file_sha256']
        score=aligned_mae(p,C_TARGET,np.load(resolve_artifact(record['row_id_file'])),C_ROW_IDS)
        assert abs(score-record['C_mae'])<1e-10
    return record

def reuse_legacy_all(candidate,identity,paths):
    name=candidate['legacy_name']
    path=OLD/'final_all'/('record_'+name+'.json')
    prior=json.loads(path.read_text())
    assert prior['recipe']==candidate['recipe']
    assert prior['identity']==canonical_hash(dict(recipe=candidate['recipe'],
        A_hash=candidate['checkpoint_sha256'],rates=candidate['selected_lr_schedule'],
        all_train_hash=array_hash(BREATH_IDS),design=canonical_hash(OLD_DESIGN)))
    assert prior['training_breaths']==67905 and prior['scored_training_rows']==int((UOUT==0).sum())
    assert prior['validation_used'] is False and prior['has_independent_holdout'] is False
    checkpoint=resolve_artifact(prior['checkpoint_path'])
    assert digest(checkpoint)==prior['checkpoint_sha256']
    old_manifest=json.loads((OLD/'final_all/final_manifest.json').read_text())
    assert old_manifest['train_sha256']==DESIGN['train_sha256']
    assert old_manifest['training_breaths']==67905 and old_manifest['final_refit_has_independent_holdout'] is False
    if candidate['kind']=='legacy_tree':
        booster=lgb.Booster(model_file=str(checkpoint))
        assert booster.num_trees()==2500 and booster.feature_name()==candidate['feature_names'];del booster
    else:
        namespace=legacy_namespace()
        model,prep,payload=namespace['load_checkpoint'](checkpoint,device='cpu');del model
        assert prep['fitted_training_breaths']==67905
        assert prep['fitted_inspiratory_points']==int((UOUT==0).sum())
        assert payload['training_recipe']['selected_lr_schedule']==candidate['selected_lr_schedule']
        assert payload['training_recipe']['validation_used'] is False
        if candidate['kind']=='legacy_warm':
            base_record=json.loads((OLD/'final_all/record_gru_s20261102.json').read_text())
            assert payload['training_recipe']['source_checkpoint']['sha256']==base_record['checkpoint_sha256']
            assert digest(resolve_artifact(base_record['checkpoint_path']))==base_record['checkpoint_sha256']
    result=dict(name=candidate['name'],stage='all',kind=candidate['kind'],status='complete',
        identity=identity,identity_sha256=canonical_hash(identity),recipe=candidate['recipe'],
        feature_mode='causal20',feature_names=candidate['feature_names'],training_breaths=67905,
        scored_training_rows=int((UOUT==0).sum()),fit_group_sha256=identity['fit_group_sha256'],
        validation_feedback=False,has_independent_holdout=False,checkpoint_path=str(checkpoint),
        checkpoint_sha256=digest(checkpoint),reused_legacy_all=True,elapsed_seconds=0.,
        provenance=dict(record_file=str(path),record_sha256=digest(path),
            final_manifest_file=str(OLD/'final_all/final_manifest.json'),
            final_manifest_sha256=digest(OLD/'final_all/final_manifest.json')))
    write_json(paths['record'],result)
    event('verified_legacy_all_reused',member=candidate['name'],checkpoint_sha256=result['checkpoint_sha256'])
    return result

def predict_member(candidate,record,raw,mask=None):
    checkpoint=resolve_artifact(record['checkpoint_path'])
    assert digest(checkpoint)==record['checkpoint_sha256']
    if candidate['kind'] in ['v5_tree','legacy_tree']:
        booster=lgb.Booster(model_file=str(checkpoint))
        assert booster.feature_name()==candidate['feature_names']
        features=np.ascontiguousarray(raw[mask] if mask is not None else raw.reshape(-1,raw.shape[-1]))
        prediction=booster.predict(features,num_iteration=int(candidate['fixed_trees']),num_threads=4)
        del booster,features
        return np.asarray(prediction,dtype=np.float64) if mask is not None else np.asarray(prediction,dtype=np.float64).reshape(raw.shape[:2])
    function=predict_checkpoint if candidate['kind']=='v5_nn' else legacy_namespace()['predict_checkpoint']
    prediction=function(checkpoint,raw,device='cuda',batch_size=512)
    return np.asarray(prediction[mask] if mask is not None else prediction,dtype=np.float64)

def fit_member(name,stage):
    assert stage in ['C','all']
    candidate=CANDIDATES[name]
    identity=member_identity(candidate,stage)
    memory_key=(stage,name)
    if memory_key in FITTED_RECORDS:
        return verify_fitted(FITTED_RECORDS[memory_key],identity,stage)
    paths=member_paths(stage,name,candidate['kind'])
    if paths['record'].exists():
        assert RESUME,'Completed fit exists; explicitly resume this confirmation round'
        result=verify_fitted(json.loads(paths['record'].read_text()),identity,stage)
        FITTED_RECORDS[memory_key]=result
        return result
    if stage=='all' and candidate['kind'].startswith('legacy_'):
        result=reuse_legacy_all(candidate,identity,paths)
        FITTED_RECORDS[memory_key]=result
        return result
    indices=GROUPS['C_fit'] if stage=='C' else ALL
    sequence,names=train_features(candidate['feature_mode'])
    assert names==candidate['feature_names']
    began=time.monotonic()
    event('fixed_fit_started',stage=stage,member=name,kind=candidate['kind'],
        training_breaths=len(indices),validation_feedback=False,identity_sha256=canonical_hash(identity))
    kind=candidate['kind']; report=None
    if kind=='v5_nn':
        last=paths['checkpoint'].with_name(paths['checkpoint'].stem+'.last'+paths['checkpoint'].suffix)
        resume=last.exists()
        assert not resume or RESUME,'Partial V5 state requires explicit resumption'
        if not resume:
            preserve_partial([paths['checkpoint'],paths['report'],paths['epochs']],'partial_'+stage+'_'+name)
        rates=candidate['selected_lr_schedule']
        assert len(rates)==candidate['best_epoch'] and len(rates)>0
        report=fit_fixed(sequence[indices],Y[indices],UOUT[indices],candidate['recipe']['model_spec'],
            candidate['recipe']['train_cfg'],candidate['seed'],rates,paths['checkpoint'],
            epoch_log_path=paths['epochs'],feature_names=names,device='cuda',resume=resume)
        write_json(paths['report'],report)
        if not report['completed']:
            event('fixed_fit_incomplete',stage=stage,member=name,executed_epochs=report['executed_epochs'])
            raise RuntimeError('Partial fixed fit saved for exact resumption')
        assert report['executed_epochs']==len(rates) and report['validation_used'] is False
    elif kind in ['legacy_base','legacy_warm']:
        assert stage=='C','All-labeled legacy models must use verified original final checkpoints'
        namespace=legacy_namespace()
        preserve_partial([paths['checkpoint'],paths['report'],paths['epochs'],paths['prediction']],'legacy_refit_'+name)
        start=None
        if kind=='legacy_warm':
            base=fit_member('legacy__gru_s20261102','C')
            assert base['fit_group_sha256']==GROUP_HASHES['C_fit'] and base['training_breaths']==54323
            start=base['checkpoint_path']
        rates=candidate['selected_lr_schedule']
        assert (len(rates)==99 if kind=='legacy_base' else rates==[1e-5]*29)
        report=namespace['fit_fixed'](sequence[indices],Y[indices],UOUT[indices],
            candidate['recipe']['model_spec'],candidate['recipe']['train_cfg'],candidate['seed'],
            rates,paths['checkpoint'],start_checkpoint=start,epoch_log_path=paths['epochs'],device='cuda')
        assert report['validation_used'] is False and report['executed_epochs']==len(rates)
        write_json(paths['report'],_json_safe(report))
    else:
        assert kind in ['v5_tree','legacy_tree']
        preserve_partial([paths['checkpoint'],paths['report'],paths['epochs'],paths['prediction'],paths['checkpoint'].with_name(paths['checkpoint'].name+'.tmp.txt')],'tree_refit_'+stage+'_'+name)
        params=copy.deepcopy(candidate['recipe']['fit_params'] if kind=='v5_tree' else candidate['legacy_tree_parameters'])
        params['n_estimators']=int(candidate['fixed_trees']);params['n_jobs']=4
        assert params['n_estimators']==2500 and params['num_leaves']==511 and params['objective']=='regression_l1'
        mask=UOUT[indices]==0
        trainx=np.ascontiguousarray(sequence[indices][mask],dtype=np.float32)
        trainy=Y[indices][mask]
        learner=lgb.LGBMRegressor(**params)
        def progress(environment):
            count=environment.iteration+1
            if count%250==0:
                _epoch_event(paths['epochs'],dict(event='trees',iteration=count,validation_used=False,
                    elapsed_seconds=time.monotonic()-began))
        progress.order=10;progress.before_iteration=False
        learner.fit(trainx,trainy,feature_name=names,callbacks=[progress])
        assert learner.booster_.num_trees()==params['n_estimators']
        temporary=paths['checkpoint'].with_name(paths['checkpoint'].name+'.tmp.txt')
        learner.booster_.save_model(str(temporary),num_iteration=params['n_estimators'])
        os.replace(temporary,paths['checkpoint'])
        report=dict(completed=True,validation_used=False,fixed_trees=params['n_estimators'],
            fit_parameters=params,training_breaths=len(indices),selected_using='A only')
        write_json(paths['report'],report)
        del learner,trainx,trainy;gc.collect()
    result=dict(name=name,stage=stage,kind=kind,status='complete',identity=identity,
        identity_sha256=canonical_hash(identity),recipe=candidate['recipe'],
        feature_mode=candidate['feature_mode'],feature_names=names,
        training_breaths=len(indices),scored_training_rows=int((UOUT[indices]==0).sum()),
        fit_group_sha256=identity['fit_group_sha256'],validation_feedback=False,
        has_independent_holdout=stage=='C',checkpoint_path=str(paths['checkpoint']),
        checkpoint_sha256=digest(paths['checkpoint']),report_file=str(paths['report']),
        report_sha256=digest(paths['report']),elapsed_seconds=time.monotonic()-began,
        selected_lr_schedule=candidate['selected_lr_schedule'],fixed_trees=candidate['fixed_trees'])
    if stage=='C':
        prediction=predict_member(candidate,result,sequence[C_IDS],C_MASK)
        score=aligned_mae(prediction,C_TARGET,C_ROW_IDS,C_ROW_IDS)
        save_array(paths['prediction'],prediction)
        result.update(C_mae=score,prediction_file=str(paths['prediction']),
            prediction_file_sha256=digest(paths['prediction']),prediction_array_sha256=array_hash(prediction),
            row_id_file=str(OUT/'C_row_ids.npy'),row_id_file_sha256=digest(OUT/'C_row_ids.npy'))
    result['elapsed_seconds']=time.monotonic()-began
    write_json(paths['record'],result)
    FITTED_RECORDS[memory_key]=result
    event('fixed_fit_completed',stage=stage,member=name,seconds=result['elapsed_seconds'],
        checkpoint_sha256=result['checkpoint_sha256'],C_mae=result.get('C_mae'))
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    return result

## Frozen C confirmation and eligibility filtering

In [ ]:
def paired_interval(first,second,mask,repeats=1000):
    counts=mask.sum(axis=1)
    assert (counts>0).all()
    difference=np.asarray(first)-np.asarray(second)
    starts=np.r_[0,np.cumsum(counts)[:-1]]
    sums=np.add.reduceat(difference,starts)
    rng=np.random.default_rng(20261121);draws=[]
    for offset in range(0,repeats,100):
        take=rng.integers(0,len(counts),size=(min(100,repeats-offset),len(counts)))
        draws.extend((sums[take].sum(axis=1)/counts[take].sum(axis=1)).tolist())
    low,high=np.quantile(draws,[.025,.975])
    return dict(difference_first_minus_second=float(difference.mean()),bootstrap_95_low=float(low),
        bootstrap_95_high=float(high),repeats=repeats,unit='whole breath',point_count_weighted=True)

if RUN and LIMIT in ['C','final']:
    assert torch.cuda.is_available(),'C neural fits require the course CUDA environment'
    assert FROZEN['lock_sha256']==json.loads((OUT/'locked_before_C.json').read_text())['lock_sha256']
    C_IDS=GROUPS['C_validation'];C_MASK=UOUT[C_IDS]==0
    C_TARGET=Y[C_IDS][C_MASK].astype(np.float64);C_ROW_IDS=ROW_IDS[C_IDS][C_MASK]
    if (OUT/'C_row_ids.npy').exists():assert np.array_equal(np.load(OUT/'C_row_ids.npy'),C_ROW_IDS)
    else:save_array(OUT/'C_row_ids.npy',C_ROW_IDS)
    C_RECORDS={name:fit_member(name,'C') for name in FROZEN['C_required_members']}
    C_PREDICTIONS={name:np.load(resolve_artifact(r['prediction_file']),allow_pickle=False) for name,r in C_RECORDS.items()}
    table,selected=choose_C(FROZEN['approaches'],C_PREDICTIONS,C_TARGET)
    def approach_prediction(name):
        a=FROZEN['approaches'][name]
        return sum(w*C_PREDICTIONS[n] for n,w in zip(a['members'],a['weights']))
    reference_error=np.abs(approach_prediction('reference')-C_TARGET)
    individual_error=np.abs(approach_prediction('best_individual')-C_TARGET)
    pair_error=np.abs(approach_prediction('best_pair')-C_TARGET)
    selection=dict(**selected,status='complete',lock_sha256=FROZEN['lock_sha256'],
        design_sha256=canonical_hash(DESIGN),rule=PLAN['rule'],table=table,
        C_records={n:dict(record_file=str(member_paths('C',n,CANDIDATES[n]['kind'])['record']),
            record_sha256=digest(member_paths('C',n,CANDIDATES[n]['kind'])['record']),
            checkpoint_sha256=r['checkpoint_sha256'],prediction_array_sha256=r['prediction_array_sha256'])
            for n,r in C_RECORDS.items()},
        paired_individual_vs_reference=paired_interval(individual_error,reference_error,C_MASK),
        paired_pair_vs_individual=paired_interval(pair_error,individual_error,C_MASK),
        weights_selected_on='A only',C_weights_tuned=False,C_is_resampling_confirmation=True,
        validation_evidence_is_from_prior_models=True,final_refit_has_independent_holdout=False)
    selection_path=OUT/'validation_selection.json'
    if selection_path.exists():assert json.loads(selection_path.read_text())==selection
    else:write_json(selection_path,selection)
    pd.DataFrame(table).to_csv(OUT/'approach_comparison.csv',index=False)
    event('C_SELECTION_COMPLETED',selection=selected,lock_sha256=FROZEN['lock_sha256'])
    display(pd.DataFrame(table))

## Final all-approved-labeled fits and exactly aligned prediction CSV

In [ ]:
if RUN and LIMIT=='final':
    selection_path=OUT/'validation_selection.json'
    selection=json.loads(selection_path.read_text())
    assert selection['status']=='complete' and selection['lock_sha256']==FROZEN['lock_sha256']
    FINAL_RECORDS={name:fit_member(name,'all') for name in selection['members']}
    CACHE.clear()
    assert digest(DATA/'test.csv')==quality['files']['test.csv']['sha256']
    sample_sha=digest(DATA/'sample_submission.csv')
    assert sample_sha=='de4c137a22804654b597723f32470b24f23d9ef662205eca0fe31b68970f0594'
    test=pd.read_csv(DATA/'test.csv',dtype=DTYPES)
    test_breath_ids=_validate_feature_frame_v005(test)
    assert not np.intersect1d(test_breath_ids,BREATH_IDS).size
    assert len(test)==603600 and test.id.is_unique
    prediction=np.zeros(len(test),dtype=np.float64)
    for name,weight in zip(selection['members'],selection['weights']):
        candidate=CANDIDATES[name]
        raw,names,ids=CACHE.get(test,candidate['feature_mode'],'approved_test')
        assert names==candidate['feature_names'] and np.array_equal(ids,test_breath_ids)
        prediction+=float(weight)*predict_member(candidate,FINAL_RECORDS[name],raw).ravel()
    sample=pd.read_csv(DATA/'sample_submission.csv')
    assert list(sample.columns)==['id','pressure'] and sample.id.is_unique
    assert len(sample)==603600 and set(sample.id)==set(test.id)
    ordered=pd.Series(prediction,index=test.id.to_numpy()).reindex(sample.id).to_numpy()
    assert ordered.shape==(603600,) and np.isfinite(ordered).all()
    FINAL.mkdir(parents=True,exist_ok=True)
    submission=FINAL/'submission.csv'
    temporary=FINAL/'submission.csv.tmp'
    pd.DataFrame(dict(id=sample.id.to_numpy(),pressure=ordered)).to_csv(temporary,index=False,float_format='%.8f')
    check=pd.read_csv(temporary)
    assert list(check.columns)==['id','pressure'] and np.array_equal(check.id.to_numpy(),sample.id.to_numpy())
    assert len(check)==603600 and np.isfinite(check.pressure.to_numpy()).all()
    assert np.allclose(check.pressure.to_numpy(),ordered,atol=1e-8,rtol=0)
    if submission.exists():
        assert RESUME and digest(submission)==digest(temporary),'Existing final CSV differs; preserve and inspect'
        temporary.unlink()
    else:os.replace(temporary,submission)
    manifest=dict(status='complete',round=ROUND,lock_file=str(OUT/'locked_before_C.json'),
        lock_sha256=FROZEN['lock_sha256'],design_sha256=canonical_hash(DESIGN),
        training_group='all approved labeled breaths',training_breaths=67905,
        scored_training_points=int((UOUT==0).sum()),members=selection['members'],weights=selection['weights'],
        model_files={n:r['checkpoint_path'] for n,r in FINAL_RECORDS.items()},
        model_hashes={n:r['checkpoint_sha256'] for n,r in FINAL_RECORDS.items()},
        model_records={n:str(member_paths('all',n,CANDIDATES[n]['kind'])['record']) for n in selection['members']},
        feature_modes={n:CANDIDATES[n]['feature_mode'] for n in selection['members']},
        recipes={n:CANDIDATES[n]['recipe'] for n in selection['members']},
        prior_A_mae=selection['A_mae'],prior_C_mae=selection['C_mae'],
        validation_evidence_file=str(selection_path),validation_evidence_sha256=digest(selection_path),
        validation_scores_are_from_prior_validation_models=True,final_refit_has_independent_holdout=False,
        train_sha256=DESIGN['train_sha256'],test_sha256=digest(DATA/'test.csv'),sample_sha256=sample_sha,
        submission_file=str(submission),submission_rows=603600,submission_sha256=digest(submission),uploaded=False)
    manifest_path=FINAL/'final_manifest.json'
    if manifest_path.exists():assert json.loads(manifest_path.read_text())==manifest
    else:write_json(manifest_path,manifest)
    event('FINAL_CSV_VERIFIED',rows=603600,sha256=manifest['submission_sha256'],uploaded=False)
    print('Verified final CSV:',submission)
    display(manifest)

## Read-only review of saved round artifacts

In [ ]:
def valid_saved_round_lock():
    lock_path=OUT/'locked_before_C.json';design_path=OUT/'design.json'
    if not lock_path.is_file() or not design_path.is_file():return None
    try:
        frozen=json.loads(lock_path.read_text());saved_design=json.loads(design_path.read_text())
        assert isinstance(frozen,dict) and frozen
        assert frozen['lock_sha256']==canonical_hash({k:v for k,v in frozen.items() if k!='lock_sha256'})
        assert frozen['design_sha256']==canonical_hash(saved_design)
        assert saved_design['source_code_sha256']==code_hash(PROJECT/'confirmation_final_v005.ipynb')
        assert set(frozen['approaches'])=={'reference','best_individual','best_pair'}
        assert frozen['selected_using'].startswith('A only')
        assert frozen['pair_weights_tuned_on_C'] is False
        assert set(frozen['C_selection_universe'])==set(frozen['approaches'])
        required=set()
        for a in frozen['approaches'].values():
            assert a['members'] and len(a['members'])==len(a['weights'])
            assert np.isfinite(a['weights']).all() and min(a['weights'])>=0
            assert abs(sum(a['weights'])-1)<1e-12
            required.update(a['members'])
        if 'legacy__warm_current' in required:required.add('legacy__gru_s20261102')
        assert required==set(frozen['C_required_members'])
        assert required.issubset(frozen['candidates'])
        return frozen
    except (AssertionError,KeyError,ValueError,TypeError,json.JSONDecodeError):return None

if not RUN:
    frozen=valid_saved_round_lock()
    if frozen is None:
        print('No valid current-round A lock; confirmation outcomes are not opened.')
    else:
        display({k:frozen[k] for k in ['lock_sha256','A_snapshot','approaches','C_required_members','rule']})
        for filename in ['validation_selection.json','final_all/final_manifest.json']:
            path=OUT/filename
            if path.exists():
                value=json.loads(path.read_text());assert value['lock_sha256']==frozen['lock_sha256']
                print(filename);display(value)